# Nighttime Indoor Environment and Sleep Outcomes Among Outdoor Workers in Non-Air-Conditioned Dormitories

- **Project:** HEATS — Cooling Dorms
- **Sites:** Dormitory A & Dormitory B, Singapore
- **Author:** Raagavi Mani, 2026
- **Data:** all source files used by this notebook ship in `data/` alongside it
- **Figures:** generated plots are written to `outputs/figures/`

## This notebook reproduces every analysis reported in the Methods and Results sections of the manuscript, in the order the manuscript describes them:

1. **Environmental data** — load Dormitory A & B sensor logs, derive absolute humidity,
   wet bulb temperature.
2. **Figures 1 and S1** — 24-hour indoor/outdoor profiles and nighttime boxplots for
   temperature, humidity, wet bulb temperature, CO₂ and PM₂.₅.
3. **Table 1 / Tables S2–S4** — descriptive statistics of the indoor dormitory environment during nighttime sleep periods.
4. **Sleep data pipeline** — actigraphy → subjective-survey merge → environmental-exposure
   attachment → data-cleaning/exclusion cascade (*Methods → Sleep Measurement, Subjective
   Surveys, Statistical Analysis*), producing the final analytic sample of **686 nights from
   38 participants**.
5. **Assumption checks** — normality, threshold exceedance, collinearity/VIF among
   exposures, and a linear-vs-spline check for nonlinearity.
6. **Table S1 / Figure 2** — within-subject OLS regressions of sleep outcomes on each
   environmental exposure, with subject-clustered SEs, room fixed effects and Bonferroni
   correction (*Methods → Statistical Analysis*).
7. **Figure 3** — thermal sensation, thermal preference and air-movement preference ratings.

## 1. Setup

In [ ]:
import os
from datetime import datetime

# ----------------------------------------------------------------------------
# All source data used below ships in the "data/" folder next to this notebook.
# All generated figures are written to "outputs/figures/" (created if missing).
# ----------------------------------------------------------------------------
DATA_DIR = "data"
FIGURES_DIR = os.path.join("outputs", "figures")
TABLES_DIR = os.path.join("outputs", "tables")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

today_date = datetime.now().strftime('%d %b %Y')
current_time = datetime.now().strftime('%H:%M:%S')
print(f"Notebook run started {today_date}, {current_time}")

: 

## 2. Environmental Data — Loading & Derived Metrics

*Methods → Environmental Monitoring.* Indoor and outdoor temperature, relative humidity,
CO₂ and PM₂.₅ were logged at 1-minute intervals by Atmocube monitors. From calibrated
temperature and relative humidity we derive:

- **Wet bulb temperature** (Stull's formula)
- **Absolute humidity** — vapor pressure from `pythermalcomfort`'s `psy_ta_rh`, converted to
  g·m⁻³ via the ideal-gas law (`abs_humidity_c`). This is the only absolute-humidity value
  used anywhere downstream: the raw on-device `abs_humidity` estimate that Atmocube also
  ships is dropped immediately below and never enters any table, plot or regression in this
  notebook.

Each dormitory ran an "INT" (intervention) and a "CON" (control) arm; only the **CON**
(non-air-conditioned, no-intervention) rooms reported in the manuscript are kept below. Dorm
A's second outdoor sensor ("Outside 2", a spare/backup — only "Outside 1" is the outdoor
sensor referenced in the manuscript) and Dorm B's Room B307 (an "INT" room, so it never had
CON environmental data) are dropped at this stage too.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
from pythermalcomfort.utilities import psy_ta_rh

# Generate today's date and current time for coding sanity checks
today_date = datetime.now().strftime('%d %b %Y')
current_time = datetime.now().strftime('%H:%M:%S')  # Format hhmmss
print(f"✅ Code has run as of {today_date}, {current_time}")

# Folder containing file
dorm_A_env_file_path = os.path.join(DATA_DIR, "env_dorm_A.parquet.gzip")
dorm_B_env_file_path = os.path.join(DATA_DIR, "env_dorm_B.parquet.gzip")

# Define time range
dorm_A_start = "2025-04-26 18:00:00"
dorm_A_end = "2025-05-31 10:00:00"

dorm_B_phase1_start = "2025-09-27 18:00:00"
dorm_B_phase1_end = "2025-10-18 12:00:00"

dorm_B_phase2_start = "2025-10-25 18:00:00"
dorm_B_phase2_end = "2025-11-15 12:00:00"

# File path
dorm_A_df = pd.read_parquet(dorm_A_env_file_path, engine="pyarrow")
dorm_B_df = pd.read_parquet(dorm_B_env_file_path, engine="pyarrow")

# Calculate wet globe temperature and add as a new column in-place

TEMP_COL = "temperature_calibrated"
RH_COL = "humidity_calibrated"
OUTPUT_COL = "wet_bulb_temp"

def calculate_wet_bulb(T, RH):
    """
    Calculates wet bulb temperature using the Stull formula.

    Parameters
    ----------
    T : array-like
        Air temperature in Celsius
    RH : array-like
        Relative humidity in %

    Returns
    -------
    np.ndarray
        Wet bulb temperature in Celsius
    """

    T = np.asarray(T, dtype=float)
    RH = np.asarray(RH, dtype=float)

    term1 = T * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
    term2 = np.arctan(T + RH)
    term3 = np.arctan(RH - 1.676331)
    term4 = 0.00391838 * (RH ** 1.5) * np.arctan(0.023101 * RH)

    wet_bulb = term1 + term2 - term3 + term4 - 4.686035

    return wet_bulb

def add_wet_bulb_column_inplace(
    df,
    temp_col=TEMP_COL,
    rh_col=RH_COL,
    output_col=OUTPUT_COL
):
    df[output_col] = np.nan

    missing_cols = [c for c in [temp_col, rh_col] if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    valid = df[temp_col].notna() & df[rh_col].notna()

    if valid.any():
        df.loc[valid, output_col] = calculate_wet_bulb(
            T=df.loc[valid, temp_col].astype(float).to_numpy(),
            RH=df.loc[valid, rh_col].astype(float).to_numpy()
        )

add_wet_bulb_column_inplace(dorm_A_df)
add_wet_bulb_column_inplace(dorm_B_df)

# Calculate abs_humdity_c and add as a new column in-place
TEMP_COL = "temperature_calibrated"
RH_COL = "humidity_calibrated"
OUTPUT_COL = "abs_humidity_c"

# ── Absolute humidity from calibrated T & RH (pythermalcomfort) ───────────────

def calculate_abs_humidity_c(T, RH):
    """
    Volumetric absolute humidity (g·m⁻³) from dry-bulb air temperature (°C)
    and relative humidity (%).

    The water-vapour partial pressure e is obtained from pythermalcomfort's
    `psy_ta_rh`; absolute humidity then follows from the ideal-gas law
    ρ_v = M_w·e / (R·T), i.e. AH[g·m⁻³] = 2.1668·e[Pa] / T[K]."""
    T = np.asarray(T, dtype="float64")
    RH  = np.asarray(RH,  dtype="float64")
    abs_humidity_c  = np.full(np.broadcast(T, RH).shape, np.nan, dtype="float64")
    tdb_b, rh_b = np.broadcast_arrays(T, RH)
    ok  = np.isfinite(tdb_b) & np.isfinite(rh_b)
    if ok.any():
        e_pa = np.asarray(
            psy_ta_rh(tdb=tdb_b[ok], rh=np.clip(rh_b[ok], 0.0, 100.0)).p_vap,
            dtype="float64")
        abs_humidity_c[ok] = 2.1668 * e_pa / (tdb_b[ok] + 273.15)
    return abs_humidity_c


def add_abs_humidity_column_inplace(
    df,
    temp_col=TEMP_COL,
    rh_col=RH_COL,
    output_col=OUTPUT_COL
):
    df[output_col] = np.nan

    missing_cols = [c for c in [temp_col, rh_col] if c not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns: {missing_cols}")

    valid = df[temp_col].notna() & df[rh_col].notna()

    if valid.any():
        df.loc[valid, output_col] = calculate_abs_humidity_c(
            T=df.loc[valid, temp_col].astype(float).to_numpy(),
            RH=df.loc[valid, rh_col].astype(float).to_numpy()
        )


add_abs_humidity_column_inplace(dorm_A_df)
add_abs_humidity_column_inplace(dorm_B_df)

# Convert index to datetime if it's not already
dorm_A_df.index = pd.to_datetime(dorm_A_df.index)
dorm_B_df.index = pd.to_datetime(dorm_B_df.index)

# Filter using a mask instead of .loc slicing for time range
dorm_A_df_filtered = dorm_A_df[(dorm_A_df.index >= dorm_A_start) & (dorm_A_df.index <= dorm_A_end)]
dorm_B_df_phase1_filtered = dorm_B_df[(dorm_B_df.index >= dorm_B_phase1_start) & (dorm_B_df.index <= dorm_B_phase1_end)]
dorm_B_df_phase2_filtered = dorm_B_df[(dorm_B_df.index >= dorm_B_phase2_start) & (dorm_B_df.index <= dorm_B_phase2_end)]

# Remove INT rooms in each of the 3 dfs
dorm_A_int_rooms_to_remove = ["B307", "A202", "A204", "A303", "A304"]  # Remaining = B205, B206, B207, Outside 1
dorm_A_unused_sensors_to_remove = ["Outside 2"]  # spare/backup outdoor sensor, not used anywhere
dorm_B_phase1_int_rooms_to_remove = ["B102", "B102 Indoor", "B102 Outdoor",
                                      "A201", "A201 Indoor", "A201 Outdoor",
                                      "B302", "B302 Indoor", "B302 Outdoor",
                                      "A104", "A305"]
    
dorm_B_phase2_int_rooms_to_remove = ["A104", "A104 Indoor", "A104 Outdoor",
                                      "A305", "A305 Indoor", "A305 Outdoor",
                                      "B102", "A201", "B302"]


dorm_A_condata_df = dorm_A_df_filtered[
    ~dorm_A_df_filtered["id_room"].isin(dorm_A_int_rooms_to_remove + dorm_A_unused_sensors_to_remove)
]

dorm_B_df_phase1_condata_df = dorm_B_df_phase1_filtered[
    ~dorm_B_df_phase1_filtered["id_room"].isin(dorm_B_phase1_int_rooms_to_remove)
]

dorm_B_df_phase2_condata_df = dorm_B_df_phase2_filtered[
    ~dorm_B_df_phase2_filtered["id_room"].isin(dorm_B_phase2_int_rooms_to_remove)
]

# View unique values
print("⭐ Dorm A Info ⭐")
print(dorm_A_condata_df["id_room"].unique())
# print(dorm_A_df_filtered["action"].unique())
# print(dorm_A_df_filtered.info())

# Create a unified CO2 column called "co2_used" for the whole dataset
if "co2_corrected" in dorm_A_condata_df.columns:
    dorm_A_condata_df["co2_used"] = dorm_A_condata_df["co2_corrected"]
elif "co2" in dorm_A_condata_df.columns:
    dorm_A_condata_df["co2_used"] = dorm_A_condata_df["co2"]
else:
    raise KeyError("dorm_A_condata_df has neither 'co2_corrected' nor 'co2'")

# print(print(dorm_A_condata_df.info()))


print("⭐ Dorm B Info Phase 1 ⭐")
print(dorm_B_df_phase1_condata_df["id_room"].unique())
# print(dorm_B_df_phase1_condata_df["action"].unique())
# print(dorm_B_df_phase1_condata_df.info())

print("⭐ Dorm B Info Phase 2 ⭐")
print(dorm_B_df_phase2_condata_df["id_room"].unique())


Dormitory B was profiled in two date-bounded phases (rooms were swapped between
"CON" and "INT" conditions between phases), so its per-phase CON subsets are combined into
one `dorm_B_condata_df` here. (Room B307 only ever ran as "INT" in Dorm A, so it is already
excluded above and does not need a corresponding entry in Dorm B's INT-room removal lists.)

In [ ]:
# Room assignments
dorm_A_room_assignments = {
    'B205' : ['CDS025', 'CDS026', 'CDS027'], #3 pax
    'B206' : ['CDS019', 'CDS029'], #2 pax
    'B207' : ['CDS030', 'CDS032', 'CDS035', 'CDS037', 'CDS038', 'CDS040'], #6 pax
}

dorm_B_phase1_con = {
    'A104': ['CDS115', 'CDS116', 'CDS117', 'CDS118', 'CDS119', 'CDS120', 'CDS121', 'CDS122', 'CD123'],
    'A305': ['CDS124', 'CDS126']
}

dorm_B_phase2_con = {
    'B102': ['CDS101', 'CDS102', 'CDS103', 'CDS128', 'CDS129', 'CDS130', 'CDS131', 'CDS132'],
    'A201': ['CDS104', 'CDS105'],
    'B302': ['CDS106', 'CDS107', 'CDS108', 'CDS109', 'CDS110', 'CDS111', 'CDS112', 'CDS113', 'CDS114']
}

tz = "Asia/Singapore"

dorm_B_phase1_start = pd.Timestamp("2025-09-27 18:00:00", tz=tz)
dorm_B_phase1_end   = pd.Timestamp("2025-10-18 12:00:00", tz=tz)

dorm_B_phase2_start = pd.Timestamp("2025-10-25 18:00:00", tz=tz)
dorm_B_phase2_end   = pd.Timestamp("2025-11-15 12:00:00", tz=tz)

phase1_con_rooms = set(dorm_B_phase1_con.keys()) #'A104', 'A305'
phase2_con_rooms = set(dorm_B_phase2_con.keys()) #'B102', 'A201', 'B302'

# Remove Phase 1 con rooms during Phase 2 dates
remove_phase1conrooms_in_phase2 = (
    dorm_B_df_phase2_filtered['id_room'].isin(phase1_con_rooms) &
    dorm_B_df_phase2_filtered.index.to_series().between(
        dorm_B_phase2_start, dorm_B_phase2_end
    )
)

# Remove Phase 2 con rooms during Phase 1 dates
remove_phase2conrooms_in_phase1 = (
    dorm_B_df_phase1_filtered['id_room'].isin(phase2_con_rooms) &
    dorm_B_df_phase1_filtered.index.to_series().between(
        dorm_B_phase1_start, dorm_B_phase1_end
    )
)

#Drop those data rows
dorm_B_df_phase2_filtered = dorm_B_df_phase2_filtered.loc[
    ~remove_phase1conrooms_in_phase2
]

dorm_B_df_phase1_filtered = dorm_B_df_phase1_filtered.loc[
    ~remove_phase2conrooms_in_phase1
]

#Combine the two phases of dorm B CON data together
dorm_B_condata_df = pd.concat(
    [dorm_B_df_phase1_filtered, dorm_B_df_phase2_filtered],
    axis=0
).sort_index()

print(f"✅ Code has run as of {today_date}, {current_time}")

#Sanity checks

#verifies data overlaps expected phase windows.  shld be same as phase start and end time
# print("Phase 1 data range:")
# print(dorm_B_df_phase1_filtered.index.min(),
#       dorm_B_df_phase1_filtered.index.max())

# print("Phase 2 data range:")
# print(dorm_B_df_phase2_filtered.index.min(),
#       dorm_B_df_phase2_filtered.index.max())

# print("RAW data range:")
# print(
#     dorm_B_df.index.min(),
#     dorm_B_df.index.max()
# )

print("All Dorm B CON data df id rooms")
# print(dorm_B_condata_df.index.min(), dorm_B_condata_df.index.max())
print(dorm_B_condata_df["id_room"].unique())

# Create a unified CO2 column for the whole dataset
if "co2_corrected" in dorm_B_condata_df.columns:
    dorm_B_condata_df["co2_used"] = dorm_B_condata_df["co2_corrected"]
elif "co2" in dorm_B_condata_df.columns:
    dorm_B_condata_df["co2_used"] = dorm_B_condata_df["co2"]
else:
    raise KeyError("dorm_B_condata_df has neither 'co2_corrected' nor 'co2'")

# print(dorm_B_condata_df.info())
# print(dorm_B_condata_df.columns)

print("Dorm A data range:")
print(
    dorm_A_condata_df.index.min(),
    dorm_A_condata_df.index.max()
)

print("Dorm B CON data range:")
print(
    dorm_B_condata_df.index.min(),
    dorm_B_condata_df.index.max()
)


####################
dorm_B_condata_df0 = dorm_B_condata_df.copy()
dorm_A_condata_df0 = dorm_A_condata_df.copy()

## 3. Indoor/Outdoor Dataset Construction

The indoor vs. outdoor sensor split, the noon-anchored hour index used for the 24-hour
profile plots, and the exclusion of one faulty-sensor night in Dormitory A are all shared by
every environmental variable plotted below (temperature, humidity, wet-bulb temperature,
CO₂, PM₂.₅). They are built once here rather than being repeated per variable.

In [ ]:
def add_hour_from_noon(df: pd.DataFrame) -> pd.DataFrame:
    """Hour-of-day index running 0-23 starting at noon (12:00 -> 0, 11:00 -> 23),
    so a 24h profile can be plotted noon-to-noon with nighttime centered."""
    df = df.copy()
    df["id_room"] = df["id_room"].astype(str)
    df["hour"] = df.index.hour
    df["hour_from_noon"] = ((df["hour"] - 12) % 24).astype(int)
    return df

DORM_A_INDOOR_ROOMS = ["B205", "B206", "B207"]
DORM_A_OUTDOOR_ROOM = "Outside 1" 

DORM_B_ROOM_LABELS = {
    "A104 Indoor": "Room A104",
    "A305 Indoor": "Room A305",
    "B102 Indoor": "Room B102",
    "A201 Indoor": "Room A201",
    "B302 Indoor": "Room B302",
}
DORM_B_INDOOR_ROOMS = list(DORM_B_ROOM_LABELS.keys())
DORM_B_OUTDOOR_ROOMS = [r.replace("Indoor", "Outdoor") for r in DORM_B_INDOOR_ROOMS]

# ---- Dorm A: split indoor vs. outdoor, drop one night with a known sensor fault ----
dorm_A = add_hour_from_noon(dorm_A_condata_df)
_faulty_night = (
    (dorm_A.index >= pd.Timestamp("2025-05-06 19:00", tz="Asia/Singapore")) &
    (dorm_A.index < pd.Timestamp("2025-05-07 07:00", tz="Asia/Singapore"))
)
dorm_A_f = dorm_A[
    dorm_A["id_room"].isin(DORM_A_INDOOR_ROOMS + [DORM_A_OUTDOOR_ROOM]) & ~_faulty_night
].copy()

dorm_A_indoor_raw = dorm_A_f[dorm_A_f["id_room"].isin(DORM_A_INDOOR_ROOMS)].copy()
dorm_A_outdoor_raw = dorm_A_f[dorm_A_f["id_room"] == DORM_A_OUTDOOR_ROOM].copy()

# ---- Dorm B: recombine both profiling phases, then split indoor vs. outdoor ----
dorm_B_condata_df = pd.concat([dorm_B_df_phase1_filtered, dorm_B_df_phase2_filtered]).sort_index()
if "co2_corrected" in dorm_B_condata_df.columns:
    dorm_B_condata_df["co2_used"] = dorm_B_condata_df["co2_corrected"]
elif "co2" in dorm_B_condata_df.columns:
    dorm_B_condata_df["co2_used"] = dorm_B_condata_df["co2"]

dorm_B = add_hour_from_noon(dorm_B_condata_df)

dorm_B_f = dorm_B[dorm_B["id_room"].isin(DORM_B_INDOOR_ROOMS + DORM_B_OUTDOOR_ROOMS)].copy()
dorm_B_f["io"] = dorm_B_f["id_room"].apply(
    lambda s: "Indoor" if "Indoor" in s else ("Outdoor" if "Outdoor" in s else "Unknown")
)

dorm_B_indoor_raw = dorm_B_f[dorm_B_f["io"] == "Indoor"].copy()
dorm_B_outdoor_raw = dorm_B_f[dorm_B_f["io"] == "Outdoor"].copy()

print(f"Dorm A — indoor rows: {len(dorm_A_indoor_raw)}, outdoor rows: {len(dorm_A_outdoor_raw)}")
print(f"Dorm B — indoor rows: {len(dorm_B_indoor_raw)}, outdoor rows: {len(dorm_B_outdoor_raw)}")

## 4. Reusable Plotting Helpers

Every panel of Figure 1 and Figure S1 is one of two chart types — a 24-hour indoor/outdoor
mean profile, or a nighttime indoor/outdoor boxplot — repeated across six environmental
variables. `plot_profile()` and `plot_boxplot()` implement each chart type once; the actual
per-variable cells below are then a few lines each.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

ALPHA = 0.05
MIN_N_PER_GROUP = 5  # require at least this many samples per hour per group to test

ORANGE = "#FFA500"   # Indoor
BLUE = "#0072B2"      # Outdoor
BLACK = "black"
DARK_BLUE = "#021923"

TITLE_FONT_SIZE = 32
AXIS_TITLE_SIZE = 30
TICK_FONT_SIZE = 26
LEGEND_FONT_SIZE = 26
NIGHTTIME_FONT_SIZE = 26

SHOW_SD_BANDS = True
SD_BAND_OPACITY = 0.18

# 24h profile x-axis: noon -> 11am, labelled every 3 hours
TICKVALS_NOON = [0, 3, 6, 9, 12, 15, 18, 21]
TICKTEXT_NOON = ["12", "15", "18", "21", "0", "3", "6", "9"]


# ---- shared helpers: 24h profile ----
def get_hourly_samples(df, value_col, hour_col="hour_from_noon"):
    """Per-hour arrays of non-null values (0-23), for per-hour significance testing."""
    return {h: df.loc[df[hour_col] == h, value_col].dropna().astype(float).values for h in range(24)}


def hourly_mean_line(df, value_col, hour_col="hour_from_noon"):
    return (
        df[[hour_col, value_col]]
        .dropna(subset=[value_col])
        .groupby(hour_col)[value_col]
        .agg(mean="mean", sd="std")
        .reindex(range(24))
        .reset_index()
        .rename(columns={hour_col: "hour_from_noon", "mean": value_col})
    )


def average_hourly_lines(df1, df2, value_col):
    """Average Dorm A's and Dorm B's hourly mean lines into one overall line."""
    merged = (
        df1[["hour_from_noon", value_col, "sd"]]
        .rename(columns={value_col: "y1", "sd": "sd1"})
        .merge(
            df2[["hour_from_noon", value_col, "sd"]].rename(columns={value_col: "y2", "sd": "sd2"}),
            on="hour_from_noon", how="outer",
        )
        .sort_values("hour_from_noon")
    )
    merged[value_col] = merged[["y1", "y2"]].mean(axis=1)
    merged["sd"] = merged[["sd1", "sd2"]].mean(axis=1)
    return merged[["hour_from_noon", value_col, "sd"]]


def welch_sig_hours(samples_a, samples_b, alpha=ALPHA, min_n=MIN_N_PER_GROUP):
    """Per-hour Welch t-test between Dorm A and Dorm B (diagnostic only; not plotted)."""
    sig_hours, pvals = [], {}
    for h in range(24):
        a, b = samples_a[h], samples_b[h]
        if len(a) >= min_n and len(b) >= min_n:
            _, p = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")
            pvals[h] = p
            if p < alpha:
                sig_hours.append(h)
        else:
            pvals[h] = np.nan
    return sig_hours, pvals


def add_sd_band(fig, df, color, name, value_col):
    band_df = df.dropna(subset=[value_col, "sd"]).copy()
    x = band_df["hour_from_noon"]
    y_upper = band_df[value_col] + band_df["sd"]
    y_lower = band_df[value_col] - band_df["sd"]
    fig.add_trace(go.Scatter(
        x=pd.concat([x, x[::-1]]), y=pd.concat([y_upper, y_lower[::-1]]),
        fill="toself", fillcolor=color, line=dict(color="rgba(255,255,255,0)"),
        hoverinfo="skip", showlegend=False, opacity=SD_BAND_OPACITY, name=f"{name} ± SD",
    ))


def plot_profile(poi, y_axis_title, y_range=None, y_dtick=None, filename=None, label=None):
    """24-hour mean indoor vs outdoor profile (noon-to-noon), Dorm A + Dorm B combined,
    with nighttime (19:00-07:00) shading. Prints per-hour Dorm A vs Dorm B significance
    (Welch t-test) as a diagnostic; this is not annotated on the figure itself."""
    label = label or poi
    indoor_a_samples = get_hourly_samples(dorm_A_indoor_raw, poi)
    indoor_b_samples = get_hourly_samples(dorm_B_indoor_raw, poi)
    outdoor_a_samples = get_hourly_samples(dorm_A_outdoor_raw, poi)
    outdoor_b_samples = get_hourly_samples(dorm_B_outdoor_raw, poi)

    indoor_a = hourly_mean_line(dorm_A_indoor_raw, poi)
    indoor_b = hourly_mean_line(dorm_B_indoor_raw, poi)
    outdoor_a = hourly_mean_line(dorm_A_outdoor_raw, poi)
    outdoor_b = hourly_mean_line(dorm_B_outdoor_raw, poi)

    indoor_overall = average_hourly_lines(indoor_a, indoor_b, poi)
    outdoor_overall = average_hourly_lines(outdoor_a, outdoor_b, poi)

    sig_hours_indoor, _ = welch_sig_hours(indoor_a_samples, indoor_b_samples)
    sig_hours_outdoor, _ = welch_sig_hours(outdoor_a_samples, outdoor_b_samples)
    print(f"[{label}] Dorm A vs Dorm B significant hours — Indoor: {sig_hours_indoor}")
    print(f"[{label}] Dorm A vs Dorm B significant hours — Outdoor: {sig_hours_outdoor}")

    fig = go.Figure()
    if SHOW_SD_BANDS:
        add_sd_band(fig, outdoor_overall, BLUE, "Outdoor", poi)
        add_sd_band(fig, indoor_overall, ORANGE, "Indoor", poi)

    fig.add_trace(go.Scatter(
        x=outdoor_overall["hour_from_noon"], y=outdoor_overall[poi], mode="lines+markers",
        name="Outdoor", line=dict(color=BLUE, width=6), marker=dict(size=11, symbol="circle"),
    ))
    fig.add_trace(go.Scatter(
        x=indoor_overall["hour_from_noon"], y=indoor_overall[poi], mode="lines+markers",
        name="Indoor", line=dict(color=ORANGE, width=6), marker=dict(size=11, symbol="circle"),
    ))

    fig.update_layout(
        template="simple_white", height=650, width=1400, margin=dict(r=260),
        font=dict(family="Arial", size=17),
        legend=dict(x=0.96, y=1, xanchor="left", yanchor="top",
                    font=dict(size=LEGEND_FONT_SIZE, family="Arial", color="black")),
    )
    fig.update_xaxes(
        title=dict(text="Hour of Day", font=dict(size=AXIS_TITLE_SIZE, family="Arial", color="black")),
        type="linear", tickmode="array", tickvals=TICKVALS_NOON, ticktext=TICKTEXT_NOON, dtick=1,
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color="black"),
        showgrid=False, showline=True, linewidth=3, linecolor="black",
        ticks="outside", ticklen=6, tickwidth=1,
    )
    fig.update_yaxes(
        title=dict(text=y_axis_title, font=dict(size=AXIS_TITLE_SIZE, family="Arial", color="black")),
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color="black"),
        showgrid=False, showline=True, linewidth=3, linecolor="black",
        ticks="outside", ticklen=6, tickwidth=1, range=y_range, dtick=y_dtick,
    )
    fig.add_vrect(x0=7, x1=19, fillcolor="rgba(0, 0, 0, 0.2)", layer="below", line_width=0)
    fig.add_annotation(
        x=13, y=1.01, xref="x", yref="paper", text="Nighttime", showarrow=False,
        font=dict(size=NIGHTTIME_FONT_SIZE, color="grey", family="Arial"),
        xanchor="center", yanchor="bottom",
    )

    fig.show()
    if filename:
        fig.write_image(os.path.join(FIGURES_DIR, filename), width=1400, height=650, scale=1)
    # No return value: fig.show() above already renders the figure. Returning fig here would
    # cause it to render a second time whenever this call is the last statement in a cell,
    # since Jupyter auto-displays an unassigned trailing expression.


# ---- shared helpers: nighttime boxplot ----
def filter_nighttime(df, value_col, hour_col="hour_from_noon"):
    """Keep 19:00-06:59 only (hour_from_noon in [7, 19))."""
    out = df[(df[hour_col] >= 7) & (df[hour_col] < 19)].copy()
    return out.dropna(subset=[value_col])


def add_night_date(df):
    """Anchor each row to the calendar date its night started (19:00-23:59 keeps that
    date; 00:00-06:59 belongs to the previous night)."""
    out = df.copy()
    ts = pd.Series(out.index.tz_localize(None), index=out.index)
    night_date = ts.dt.normalize()
    night_date = night_date.where(ts.dt.hour >= 7, night_date - pd.Timedelta(days=1))
    out["night_date"] = night_date.values
    return out


def count_nights(df):
    if "night_date" not in df.columns:
        df = add_night_date(df)
    return df["night_date"].nunique()


def box_stats(df, value_col):
    vals = df[value_col].dropna().astype(float)
    return {
        "n_samples": len(vals), "min": vals.min(), "q1": vals.quantile(0.25),
        "median": vals.quantile(0.50), "q3": vals.quantile(0.75), "max": vals.max(),
    }


def _nighttime_indoor_outdoor(poi):
    dorm_A_indoor_night = add_night_date(filter_nighttime(dorm_A_indoor_raw, poi))
    dorm_A_outdoor_night = add_night_date(filter_nighttime(dorm_A_outdoor_raw, poi))
    dorm_B_indoor_night = add_night_date(filter_nighttime(dorm_B_indoor_raw, poi))
    dorm_B_outdoor_night = add_night_date(filter_nighttime(dorm_B_outdoor_raw, poi))

    indoor_night = pd.concat([
        dorm_A_indoor_night[[poi, "night_date"]], dorm_B_indoor_night[[poi, "night_date"]]
    ])
    outdoor_night = pd.concat([
        dorm_A_outdoor_night[[poi, "night_date"]], dorm_B_outdoor_night[[poi, "night_date"]]
    ])
    night_counts = dict(
        dorm_A_indoor=count_nights(dorm_A_indoor_night), dorm_A_outdoor=count_nights(dorm_A_outdoor_night),
        dorm_B_indoor=count_nights(dorm_B_indoor_night), dorm_B_outdoor=count_nights(dorm_B_outdoor_night),
        overall_indoor=count_nights(indoor_night), overall_outdoor=count_nights(outdoor_night),
    )
    return indoor_night, outdoor_night, night_counts


def plot_boxplot(poi, y_axis_title, label_decimals=1, filename=None, label=None):
    """Nighttime (19:00-07:00) indoor vs outdoor boxplot, Dorm A + Dorm B nights combined."""
    label = label or poi
    indoor_night, outdoor_night, night_counts = _nighttime_indoor_outdoor(poi)
    print(f"[{label}] Nights used: {night_counts}")

    indoor_stats = box_stats(indoor_night, poi)
    outdoor_stats = box_stats(outdoor_night, poi)
    print(f"[{label}] Indoor nighttime stats: {indoor_stats}")
    print(f"[{label}] Outdoor nighttime stats: {outdoor_stats}")

    fig_box = go.Figure()
    fig_box.add_trace(go.Box(
        x=[0], name="Outdoor", q1=[outdoor_stats["q1"]], median=[outdoor_stats["median"]],
        q3=[outdoor_stats["q3"]], lowerfence=[outdoor_stats["min"]], upperfence=[outdoor_stats["max"]],
        line=dict(color=DARK_BLUE, width=3), fillcolor=BLUE, opacity=0.9, boxpoints=False,
        showlegend=False, width=0.45,
    ))
    fig_box.add_trace(go.Box(
        x=[1], name="Indoor", q1=[indoor_stats["q1"]], median=[indoor_stats["median"]],
        q3=[indoor_stats["q3"]], lowerfence=[indoor_stats["min"]], upperfence=[indoor_stats["max"]],
        line=dict(color=BLACK, width=3), fillcolor=ORANGE, opacity=0.9, boxpoints=False,
        showlegend=False, width=0.45,
    ))
    fig_box.add_trace(go.Scatter(
        x=[0.28, 1.28], y=[outdoor_stats["median"], indoor_stats["median"]], mode="text",
        text=[f'{outdoor_stats["median"]:.{label_decimals}f}', f'{indoor_stats["median"]:.{label_decimals}f}'],
        textposition="middle right", textfont=dict(size=TICK_FONT_SIZE, color="black"), showlegend=False,
    ))

    all_vals = pd.concat([indoor_night[poi], outdoor_night[poi]]).dropna().astype(float)
    ymin, ymax = np.floor(all_vals.min() - 0.5), np.ceil(all_vals.max() + 0.5)

    fig_box.update_layout(
        template="plotly_white", height=650, width=800, margin=dict(r=40, l=90, t=100, b=80),
        font=dict(family="Arial", size=17, color=BLACK),
        legend=dict(x=1.2, y=0.98, xanchor="right", yanchor="top",
                    font=dict(family="Arial", size=LEGEND_FONT_SIZE, color=BLACK), bgcolor="rgba(255,255,255,0)"),
        boxmode="group",
    )
    fig_box.update_xaxes(
        title=dict(text="Setting", font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK)),
        tickmode="array", tickvals=[0, 1], ticktext=["Outdoor", "Indoor"], range=[-0.5, 1.5],
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside", ticklen=6, tickwidth=1,
    )
    fig_box.update_yaxes(
        title=dict(text=y_axis_title, font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK)),
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside", ticklen=6, tickwidth=1,
        range=[ymin, ymax],
    )

    fig_box.show()
    if filename:
        fig_box.write_image(os.path.join(FIGURES_DIR, filename), width=800, height=650, scale=1)
    # No return value -- see the note at the end of plot_profile() above.


def plot_boxplot_broken_axis(poi, y_axis_title, lower_range, upper_range, lower_tickvals,
                              upper_tickvals, label_decimals=0, filename=None, label=None):
    """Nighttime indoor vs outdoor boxplot with a broken y-axis, for variables (PM2.5) whose
    outdoor readings include occasional large spikes that would otherwise dwarf the boxes."""
    label = label or poi
    indoor_night, outdoor_night, night_counts = _nighttime_indoor_outdoor(poi)
    print(f"[{label}] Nights used: {night_counts}")

    indoor_stats = box_stats(indoor_night, poi)
    outdoor_stats = box_stats(outdoor_night, poi)
    print(f"[{label}] Indoor nighttime stats: {indoor_stats}")
    print(f"[{label}] Outdoor nighttime stats: {outdoor_stats}")

    def add_box(fig, s, x, name, fillcolor, linecolor, row):
        fig.add_trace(go.Box(
            x=[x], name=name, q1=[s["q1"]], median=[s["median"]], q3=[s["q3"]],
            lowerfence=[s["min"]], upperfence=[s["max"]], line=dict(color=linecolor, width=3),
            fillcolor=fillcolor, opacity=0.9, boxpoints=False, showlegend=False, width=0.45,
        ), row=row, col=1)

    fig_box = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                             row_heights=[0.30, 0.70])
    add_box(fig_box, outdoor_stats, 0, "Outdoor", BLUE, DARK_BLUE, row=1)
    add_box(fig_box, indoor_stats, 1, "Indoor", ORANGE, BLACK, row=1)
    add_box(fig_box, outdoor_stats, 0, "Outdoor", BLUE, DARK_BLUE, row=2)
    add_box(fig_box, indoor_stats, 1, "Indoor", ORANGE, BLACK, row=2)

    fig_box.add_trace(go.Scatter(
        x=[0.28, 1.28], y=[outdoor_stats["median"], indoor_stats["median"]], mode="text",
        text=[f'{outdoor_stats["median"]:.{label_decimals}f}', f'{indoor_stats["median"]:.{label_decimals}f}'],
        textposition="middle right", textfont=dict(size=TICK_FONT_SIZE, color=BLACK, family="Arial"),
        showlegend=False, hoverinfo="skip",
    ), row=2, col=1)

    fig_box.update_layout(
        template="plotly_white", height=650, width=800, margin=dict(r=40, l=130, t=100, b=80),
        font=dict(family="Arial", size=17, color=BLACK), boxmode="group",
    )
    fig_box.update_xaxes(showticklabels=False, title_text="", showgrid=False, showline=False,
                          linewidth=3, linecolor=BLACK, ticks="", row=1, col=1)
    fig_box.update_xaxes(
        title=dict(text="Setting", font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK)),
        tickmode="array", tickvals=[0, 1], ticktext=["Outdoor", "Indoor"], range=[-0.5, 1.5],
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside",
        ticklen=6, tickwidth=1, row=2, col=1,
    )
    fig_box.update_yaxes(range=upper_range, tickvals=upper_tickvals,
                          tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
                          showgrid=False, showline=True, linewidth=3, linecolor=BLACK,
                          ticks="outside", ticklen=6, tickwidth=1, row=1, col=1)
    fig_box.update_yaxes(
        range=lower_range, tickvals=lower_tickvals,
        tickfont=dict(size=TICK_FONT_SIZE, family="Arial", color=BLACK),
        showgrid=False, showline=True, linewidth=3, linecolor=BLACK, ticks="outside",
        ticklen=6, tickwidth=1, row=2, col=1,
    )
    fig_box.add_annotation(x=-0.50, y=lower_range[1], xref="x2", yref="y2", text="//",
                            showarrow=False, font=dict(size=30, color=BLACK, family="Arial"))
    fig_box.add_annotation(x=-0.50, y=upper_range[0], xref="x", yref="y", text="//",
                            showarrow=False, font=dict(size=30, color=BLACK, family="Arial"))

    # A per-subplot y-axis title (row=2 only) would sit centered on the lower subplot alone,
    # not on the combined two-panel axis -- so instead add one standalone, rotated annotation
    # spanning the full figure height (paper coordinates), centered on the whole plot.
    fig_box.add_annotation(
        text=y_axis_title, textangle=-90, x=-0.145, y=0.5, xref="paper", yref="paper",
        xanchor="center", yanchor="middle", showarrow=False,
        font=dict(size=AXIS_TITLE_SIZE, family="Arial", color=BLACK),
    )

    fig_box.show()
    if filename:
        fig_box.write_image(os.path.join(FIGURES_DIR, filename), width=800, height=650, scale=0.8)
    # No return value -- see the note at the end of plot_profile() above.

## 5. Environmental Profiles — Figure 1 & Figure S1

Each variable below produces the same pair of panels: a 24-hour indoor/outdoor mean profile
(Fig. 1a-d, Fig. S1a-b) and a nighttime indoor/outdoor boxplot (Fig. 1e-h, Fig. S1c-d).

### 5.1 Temperature (Figure 1a, 1e)

In [ ]:
plot_profile(
    poi="temperature_calibrated", y_axis_title="Mean Temperature (°C)",
    y_range=[27, 37], y_dtick=2, filename="temperature_indoor_outdoor_profile.png",
    label="Temperature",
)
plot_boxplot(
    poi="temperature_calibrated", y_axis_title="Nighttime Temperature (°C)",
    label_decimals=1, filename="temperature_indoor_outdoor_boxplot.png", label="Temperature",
)

### 5.2 Relative Humidity (Figure 1c, 1g)

In [ ]:
plot_profile(
    poi="humidity_calibrated", y_axis_title="Mean Relative Humidity (%)",
    y_dtick=10, filename="relativehumidity_indoor_outdoor_profile.png", label="Relative Humidity",
)
plot_boxplot(
    poi="humidity_calibrated", y_axis_title="Nighttime Relative Humidity (%)",
    label_decimals=0, filename="relativehumidity_indoor_outdoor_boxplot.png", label="Relative Humidity",
)

### 5.3 Absolute Humidity (Figure 1b, 1f)

In [ ]:
plot_profile(
    poi="abs_humidity_c", y_axis_title="Mean Absolute Humidity (g/m³)",
    filename="absolutehumidity_indoor_outdoor_profile.png", label="Absolute Humidity",
)
plot_boxplot(
    poi="abs_humidity_c", y_axis_title="Nighttime Absolute Humidity (g/m³)",
    label_decimals=0, filename="absolutehumidity_indoor_outdoor_boxplot.png", label="Absolute Humidity",
)

### 5.4 Wet Bulb Temperature (Figure 1d, 1h)

In [ ]:
plot_profile(
    poi="wet_bulb_temp", y_axis_title="Mean Wet Bulb Temperature (°C)",
    y_dtick=1, filename="wetbulb_indoor_outdoor_profile.png", label="Wet Bulb Temperature",
)
plot_boxplot(
    poi="wet_bulb_temp", y_axis_title="Nighttime Wet Bulb Temperature (°C)",
    label_decimals=1, filename="wetbulb_indoor_outdoor_boxplot.png", label="Wet Bulb Temperature",
)

### 5.5 CO₂ Concentration (Figure S1a, S1c)

In [ ]:
plot_profile(
    poi="co2_used", y_axis_title="Mean CO₂ Concentration (ppm)",
    filename="co2_indoor_outdoor_profile.png", label="CO2",
)
plot_boxplot(
    poi="co2_used", y_axis_title="Nighttime CO₂ Concentration (ppm)",
    label_decimals=0, filename="co2_indoor_outdoor_boxplot.png", label="CO2",
)

### 5.6 PM₂.₅ Concentration (Figure S1b, S1d)

Outdoor PM₂.₅ includes occasional large spikes (max ≈ 900 µg/m³) that would dwarf the
indoor/outdoor boxes on a linear axis, so the nighttime boxplot uses a broken y-axis. (The
original notebook also kept an unbroken-axis version of this same boxplot as a separate,
near-identical cell; it is dropped here as a duplicate — see the alignment notes.)

In [ ]:
plot_profile(
    poi="pm25", y_axis_title="Mean PM₂.₅ Concentration (μg/m³)", y_range=[0, 40], y_dtick=10,
    filename="pm25_indoor_outdoor_profile.png", label="PM2.5",
)
plot_boxplot_broken_axis(
    poi="pm25", y_axis_title="Nighttime PM₂.₅ Concentration (μg/m³)",
    lower_range=[0, 30], upper_range=[200, 1000],
    lower_tickvals=[0, 15, 30], upper_tickvals=[200, 1000],
    label_decimals=0, filename="pm25_indoor_outdoor_boxplot.png", label="PM2.5",
)

## 6. Descriptive Statistics of the Dormitory Environment

Daily (00:00-23:59) and nighttime (19:00-06:59) descriptive tables, each reported both as
mean ± SD and as median (P25-P75; P5-P95), split out by dormitory and indoor/outdoor.
Absolute humidity is `abs_humidity_c` only.

In [ ]:
import pandas as pd
import numpy as np

# ================================
# 1) Helpers
# ================================
def add_time_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds:
      - hour
      - date: calendar day, 00:00–23:59
      - night_date: anchored to start date of night at 19:00
      - is_night: night = 19:00–06:59
    """
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("DataFrame index must be a DatetimeIndex")

    df["hour"] = df.index.hour
    df["date"] = df.index.normalize()

    is_night = (df["hour"] >= 19) | (df["hour"] < 7)

    df["night_date"] = df["date"]
    df.loc[df["hour"] < 7, "night_date"] = (
        df.loc[df["hour"] < 7, "night_date"] - pd.Timedelta(days=1)
    )

    df.loc[~is_night, "night_date"] = pd.NaT
    df["is_night"] = is_night
    return df


def ensure_co2_used(df: pd.DataFrame) -> pd.DataFrame:
    """Guarantee a 'co2_used' column exists."""
    df = df.copy()
    if "co2_used" in df.columns:
        return df
    if "co2_corrected" in df.columns:
        df["co2_used"] = df["co2_corrected"]
    elif "co2" in df.columns:
        df["co2_used"] = df["co2"]
    else:
        raise KeyError("Neither 'co2_used', 'co2_corrected' nor 'co2' found")
    return df


def ensure_abs_humidity_c(df: pd.DataFrame) -> pd.DataFrame:
    """Guarantee abs_humidity_c exists."""
    df = df.copy()
    if "abs_humidity_c" not in df.columns:
        raise KeyError("Missing required column: 'abs_humidity_c'")
    return df

def format_mean_sd(x: pd.Series) -> tuple[str, int]:
    """Return 'mean ± SD', 2dp."""
    s = pd.to_numeric(x, errors="coerce").dropna()
    n = int(s.shape[0])
    if n == 0:
        return "NA", 0
    return f"{s.mean():.2f} ± {s.std():.2f}", n


def format_median_iqr_p5p95(x: pd.Series) -> tuple[str, int]:
    """Return 'median (P25–P75; P5–P95)', 2dp."""
    s = pd.to_numeric(x, errors="coerce").dropna()
    n = int(s.shape[0])
    if n == 0:
        return "NA", 0

    return (
        f"{s.median():.2f} "
        f"({s.quantile(0.25):.2f}–{s.quantile(0.75):.2f}; "
        f"{s.quantile(0.05):.2f}–{s.quantile(0.95):.2f})",
        n,
    )


def concat_pair(a: pd.DataFrame, b: pd.DataFrame) -> pd.DataFrame:
    """Concat two dataframes and sort by time index."""
    return pd.concat([a, b], axis=0).sort_index()


def aggregate_by_day(df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    """
    Aggregate to one value per calendar day, 00:00–23:59,
    using the daily mean.
    """
    d = df.copy()
    for m in metrics:
        d[m] = pd.to_numeric(d[m], errors="coerce")

    out = d.groupby("date")[metrics].mean(numeric_only=True)
    out.index = pd.to_datetime(out.index)
    return out.sort_index()


def aggregate_by_night_7pm_7am(df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    """
    Aggregate to one value per night, 19:00–06:59,
    anchored to the start date at 19:00.
    """
    d = df[df["is_night"]].copy()

    for m in metrics:
        d[m] = pd.to_numeric(d[m], errors="coerce")

    d = d.dropna(subset=["night_date"])

    out = d.groupby("night_date")[metrics].mean(numeric_only=True)
    out.index = pd.to_datetime(out.index)
    return out.sort_index()


def build_table_from_aggregated(
    dfs_agg: dict[str, pd.DataFrame],
    metrics: list[str],
    formatter,
    period_label_for_counts: str,
) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """
    Build summary table from aggregated daily or nightly values.
    """
    table = pd.DataFrame(index=metrics, columns=list(dfs_agg.keys()))
    n_obs = {}
    n_periods = {}

    for col_name, df in dfs_agg.items():
        temp_periods = int(
            pd.to_numeric(df["temperature_calibrated"], errors="coerce")
            .notna()
            .sum()
        )

        n_obs[col_name] = temp_periods
        n_periods[col_name] = temp_periods

        for m in metrics:
            cell, _ = formatter(df[m])
            table.loc[m, col_name] = cell

    metric_rename = {
        "temperature_calibrated": "Temperature (°C)",
        "humidity_calibrated": "Relative Humidity (%)",
        "abs_humidity_c": "Absolute Humidity (g/m³)",
        "wet_bulb_temp": "Wet Bulb Temperature (°C)",
        "co2_used": "CO₂ (ppm)",
        "pm25": "PM₂.₅ (µg/m³)",
    }

    table = table.rename(index=metric_rename)

    n_obs = pd.Series(
        n_obs,
        name=f"n ({period_label_for_counts} with non-null temperature)",
    )
    n_periods = pd.Series(
        n_periods,
        name=f"n_{period_label_for_counts} (temperature present)",
    )

    return table, n_obs, n_periods


# ================================
# 2) Build the datasets
# ================================

# ---- Dorm A ----
dorm_A = dorm_A_condata_df.copy()
dorm_A["id_room"] = dorm_A["id_room"].astype(str)

dorm_A = add_time_fields(dorm_A)
dorm_A = ensure_co2_used(dorm_A)
dorm_A = ensure_abs_humidity_c(dorm_A)

dorm_a_indoor_room_ids = ["B205", "B206", "B207"]
dorm_a_outdoor_id = "Outside 1"

dorm_A_indoor = dorm_A[dorm_A["id_room"].isin(dorm_a_indoor_room_ids)].copy()
dorm_A_outdoor = dorm_A[dorm_A["id_room"] == dorm_a_outdoor_id].copy()

# Remove problematic night: night anchored at 2025-05-06 19:00
exclude_night = pd.Timestamp("2025-05-06", tz="Asia/Singapore")

dorm_A_indoor = dorm_A_indoor[
    dorm_A_indoor["night_date"] != exclude_night
].copy()


# ---- Dorm B ----
dorm_B_condata_df = pd.concat(
    [dorm_B_df_phase1_filtered, dorm_B_df_phase2_filtered]
).sort_index()

dorm_B = dorm_B_condata_df.copy()
dorm_B["id_room"] = dorm_B["id_room"].astype(str)

dorm_B = add_time_fields(dorm_B)
dorm_B = ensure_co2_used(dorm_B)
dorm_B = ensure_abs_humidity_c(dorm_B)

indoor_rooms_b = [
    "A104 Indoor",
    "A305 Indoor",
    "B102 Indoor",
    "A201 Indoor",
    "B302 Indoor",
]

outdoor_rooms_b = [r.replace("Indoor", "Outdoor") for r in indoor_rooms_b]

dorm_B_f = dorm_B[
    dorm_B["id_room"].isin(indoor_rooms_b + outdoor_rooms_b)
].copy()

dorm_B_f["io"] = dorm_B_f["id_room"].apply(
    lambda s: "Indoor"
    if "Indoor" in s
    else ("Outdoor" if "Outdoor" in s else "Unknown")
)

dorm_B_f = ensure_co2_used(dorm_B_f)

dorm_B_indoor = dorm_B_f[dorm_B_f["io"] == "Indoor"].copy()
dorm_B_outdoor = dorm_B_f[dorm_B_f["io"] == "Outdoor"].copy()


# ================================
# 3) Column sets
# ================================
METRICS = [
    "temperature_calibrated",
    "humidity_calibrated",
    "abs_humidity_c",
    "wet_bulb_temp",
    "co2_used",
    "pm25",
]

COLS_EXT = [
    "Dorm A Indoor",
    "Dorm B Indoor",
    "Dorm A+B Indoor",
    "Dorm A Outdoor",
    "Dorm B Outdoor",
    "Dorm A+B Outdoor",
]


# ================================
# 4) Raw dataframes by location
# ================================
dfs_raw = {
    "Dorm A Indoor": dorm_A_indoor,
    "Dorm B Indoor": dorm_B_indoor,
    "Dorm A+B Indoor": concat_pair(dorm_A_indoor, dorm_B_indoor),
    "Dorm A Outdoor": dorm_A_outdoor,
    "Dorm B Outdoor": dorm_B_outdoor,
    "Dorm A+B Outdoor": concat_pair(dorm_A_outdoor, dorm_B_outdoor),
}


# ================================
# 5) Aggregate daily and nightly values
# ================================
dfs_day_agg = {
    k: aggregate_by_day(v, METRICS)
    for k, v in dfs_raw.items()
}

dfs_night_agg = {
    k: aggregate_by_night_7pm_7am(v, METRICS)
    for k, v in dfs_raw.items()
}


# ================================
# TABLE 1: Daily mean ± SD
# ================================
table_1, n1_obs, n1_days = build_table_from_aggregated(
    dfs_day_agg,
    METRICS,
    formatter=format_mean_sd,
    period_label_for_counts="days",
)

table_1 = table_1[COLS_EXT]

print("\nTABLE 1 — Daily values (00:00–23:59): mean ± SD, 2dp")
print(table_1.to_string())

print("\nN (days with non-null temperature) for TABLE 1:")
print(n1_obs[COLS_EXT].to_string())


# ================================
# TABLE 2: Daily median, IQR, P5–P95
# ================================
table_2, n2_obs, n2_days = build_table_from_aggregated(
    dfs_day_agg,
    METRICS,
    formatter=format_median_iqr_p5p95,
    period_label_for_counts="days",
)

table_2 = table_2[COLS_EXT]

print("\nTABLE 2 — Daily values (00:00–23:59): median (P25–P75; P5–P95), 2dp")
print(table_2.to_string())

print("\nN (days with non-null temperature) for TABLE 2:")
print(n2_obs[COLS_EXT].to_string())


# ================================
# TABLE 3: Night mean ± SD
# ================================
table_3, n3_obs, n3_nights = build_table_from_aggregated(
    dfs_night_agg,
    METRICS,
    formatter=format_mean_sd,
    period_label_for_counts="nights",
)

table_3 = table_3[COLS_EXT]

print("\nTABLE 3 — Night values (19:00–06:59): mean ± SD, 2dp")
print(table_3.to_string())

print("\nN (nights with non-null temperature) for TABLE 3:")
print(n3_obs[COLS_EXT].to_string())


# ================================
# TABLE 4: Night median, IQR, P5–P95
# ================================
table_4, n4_obs, n4_nights = build_table_from_aggregated(
    dfs_night_agg,
    METRICS,
    formatter=format_median_iqr_p5p95,
    period_label_for_counts="nights",
)

table_4 = table_4[COLS_EXT]

print("\nTABLE 4 — Night values (19:00–06:59): median (P25–P75; P5–P95), 2dp")
print(table_4.to_string())

print("\nN (nights with non-null temperature) for TABLE 4:")
print(n4_obs[COLS_EXT].to_string())

## 7. Sleep Data Pipeline

*Methods → Sleep Measurement, Subjective Surveys, Statistical Analysis.* Builds the
per-night analytic dataset in four steps, each writing an intermediate CSV to `data/` so
the pipeline can be resumed from any stage:

1. **Actigraphy sleep metrics** (Tudor-Locke-scored TIB/TST/SE/SOL/WASO from the Ametris
   watch) → `heats-dorms-sleepperiodmetrics-1h.csv`
2. **Merge subjective (Qualtrics) ratings** (thermal sensation/preference, air-movement
   preference) → `...-1h-qualtrics.csv`
3. **Attach the environmental exposure during each sleep period** → `...-1h-qualtrics-env.csv`
4. **Clean & exclude** down to the final analytic sample of 686 nights / 38 participants →
   `...-1h-qualtrics-env-clean.csv`

### 7.1 Actigraphy sleep metrics (file #1)

In [ ]:
import pandas as pd
from datetime import timedelta
from pandas.api.types import is_datetime64_any_dtype

# -------------------------------------
# Step 0: Load data
# -------------------------------------
con_sleepdata_filepath = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics.csv")
df = pd.read_csv(con_sleepdata_filepath)
# -------------------------------------
# Step 1: Define room assignments
# -------------------------------------
room_assignments = {
    'B205' : ['CDS025', 'CDS026', 'CDS027'],
    'B206' : ['CDS019', 'CDS029'],
    'B207' : ['CDS030', 'CDS032', 'CDS035', 'CDS037', 'CDS038', 'CDS040'],
    'A104': ['CDS115', 'CDS116', 'CDS117', 'CDS118', 'CDS119', 'CDS120', 'CDS121', 'CDS122', 'CDS123'],
    'A305': ['CDS124', 'CDS126'],
    'B102': ['CDS101', 'CDS102', 'CDS103', 'CDS128', 'CDS129', 'CDS130', 'CDS131', 'CDS132'],
    'A201': ['CDS104', 'CDS105'],
    'B302': ['CDS106', 'CDS107', 'CDS108', 'CDS109', 'CDS110', 'CDS111', 'CDS112', 'CDS113', 'CDS114']
}

# Reverse mapping (subject → room)
subject_to_room = {
    subject: room
    for room, subjects in room_assignments.items()
    for subject in subjects
}

# -------------------------------------
# Step 2: Filter dataset
# -------------------------------------
# Keep only rows with valid logged sleep
if df['SleepDataLogged'].dtype == 'object':
    df_filtered = df[df['SleepDataLogged'].astype(str).str.upper() == 'TRUE']
else:
    df_filtered = df[df['SleepDataLogged'] == True]

# df_filtered = df.copy() # hash this when unhashing above filter

# Keep only participants from the specified rooms
participants = [s for subs in room_assignments.values() for s in subs]
df_filtered = df_filtered[df_filtered['Subject'].isin(participants)]

# Add room column
# df_filtered['Room'] = df_filtered['Subject'].map(subject_to_room)

# -------------------------------------
# Step 3: Select and process relevant columns
# -------------------------------------
cols_to_keep = ['Subject', 'Site', 'Room', 'Adjusted_date', 'Night_WearTime_min', 'SleepDataLogged',
                'InBedTime', 'OutBedTime', 
                'Onset', 'LatencyInMinutes', 'AvgAwakeningInMinutes', 'AwakeningCount', 'Efficiency', 
                'TimeAsleepInMinutes', 'TimeAwakeInMinutes', 'WakeAfterOnsetInMinutes', 'TotalInBedTime_minutes']
df_window = df_filtered[cols_to_keep].copy()

# Multiply Efficiency column by 100
df_window["Efficiency"] = df_window["Efficiency"] * 100

# Convert to datetime (dayfirst=True since data is dd/m/yyyy)
df_window['InBedTime'] = pd.to_datetime(df_window['InBedTime'], errors='coerce', dayfirst=True)
df_window['OutBedTime'] = pd.to_datetime(df_window['OutBedTime'], errors='coerce', dayfirst=True)
df_window['Onset'] = pd.to_datetime(df_window['Onset'], errors='coerce', dayfirst=True)

# Convert Adjusted_date to yyyy-mm-dd (date only, no time)
df_window['Adjusted_date'] = pd.to_datetime(
    df_window['Adjusted_date'],
    dayfirst=True,
    errors='coerce'
).dt.date

# Drop rows with invalid timestamps
df_window = df_window.dropna(subset=['InBedTime', 'OutBedTime'])

# -------------------------------------
# Step 4: Create ±1 hour window columns
# -------------------------------------
df_window['InBedTime_1hrbefore'] = df_window['InBedTime'] - timedelta(hours=1)
# df_window['OutBedTime_1hrafter'] = df_window['OutBedTime'] + timedelta(hours=0)

# -------------------------------------
# Step 5: Create day of week column
# -------------------------------------

# Ensure Adjusted_date is datetime
df_window['Adjusted_date'] = pd.to_datetime(
    df_window['Adjusted_date'], errors='coerce'
)

# Create day-of-week column
df_window['Adjusted_day'] = df_window['Adjusted_date'].dt.day_name()

# Move Adjusted_day right after Adjusted_date
cols = df_window.columns.tolist()
idx = cols.index('Adjusted_date')
cols.insert(idx + 1, cols.pop(cols.index('Adjusted_day')))
df_window = df_window.loc[:, cols]

# -------------------------------------
# Step 5: Recalculate and Create TIB_recalculated and SE_recalculated columns
# -------------------------------------

df_window['TIB_recalculated'] = (
    pd.to_numeric(df_window['TimeAsleepInMinutes'], errors='coerce') +
    pd.to_numeric(df_window['WakeAfterOnsetInMinutes'], errors='coerce') +
    pd.to_numeric(df_window['LatencyInMinutes'], errors='coerce'))

df_window['SE_recalculated'] = (
    pd.to_numeric(df_window['TimeAsleepInMinutes'], errors='coerce')  /
    pd.to_numeric(df_window['TIB_recalculated'], errors='coerce')
) * 100

# Move SE_recalculated right after Efficiency
cols = df_window.columns.tolist()
idx_1 = cols.index('Efficiency')
cols.insert(idx_1 + 1, cols.pop(cols.index('SE_recalculated')))
# df_window = df_window.loc[:, cols]

# Move TIB_recalculated right after TotalInBedTime_minutes
idx_2 = cols.index('TotalInBedTime_minutes')
cols.insert(idx_2 + 1, cols.pop(cols.index('TIB_recalculated')))
df_window = df_window.loc[:, cols]

# -------------------------------------
# Step 6: Sort
# -------------------------------------

# Create numeric Subject sort key
df_window['_Subject_num'] = (
    df_window['Subject']
    .str.extract(r'(\d+)')
    .astype(int)
)

# Sort by Subject (numeric), then by Adjusted_date
df_window = (
    df_window
    .sort_values(by=['_Subject_num', 'Adjusted_date'])
    .drop(columns='_Subject_num')
    .reset_index(drop=True)
)

# -------------------------------------
# Step 7: Save to CSV
# -------------------------------------
output_path = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics-1h.csv")
df_window.to_csv(output_path, index=False)

# -------------------------------------
# Step 8: Summary
# -------------------------------------
print(f"✅ Code has run as of {today_date}, {current_time}")
print(f"✅ New filtered file saved as:\n{output_path}")
print(f"Total valid rows: {len(df_window)}")
print(f"Unique participants: {df_window['Subject'].nunique()}")
print(f"Rooms included: {df_window['Room'].unique().tolist()}")
print(f"Columns included: {list(df_window.columns)}")


### 7.2 Merge subjective (Qualtrics) survey ratings (file #2)

In [ ]:
import pandas as pd
from datetime import datetime

# --------------------
# Timestamp
# --------------------
today_date = datetime.today().date()
current_time = datetime.now().strftime("%H:%M:%S")

# --------------------
# File paths
# --------------------
con_sleepdata_window_filepath = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics-1h.csv")
dorm_A_qualtrics_filepath = os.path.join(DATA_DIR, "dorm_A_qualtrics.csv")
dorm_B_qualtrics_filepath = os.path.join(DATA_DIR, "dorm_B_qualtrics.csv")

output_w_qualtrics_path = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics.csv")

# --------------------
# Load data
# --------------------
sleep_df = pd.read_csv(con_sleepdata_window_filepath)
dorm_A_q = pd.read_csv(dorm_A_qualtrics_filepath)
dorm_B_q = pd.read_csv(dorm_B_qualtrics_filepath)

# --------------------
# Normalize join keys
# --------------------
for df in [sleep_df, dorm_A_q, dorm_B_q]:
    df['Subject'] = df['Subject'].astype(str).str.strip().str.upper()
    df['Adjusted_date'] = pd.to_datetime(
        df['Adjusted_date'], errors='coerce'
    ).dt.date

# --------------------
# Sanity checks
# --------------------
print("\n--- SANITY CHECKS ---")
print("Sleep rows:", len(sleep_df))
print("Dorm A rows:", len(dorm_A_q))
print("Dorm B rows:", len(dorm_B_q))

sleep_keys = set(zip(sleep_df['Subject'], sleep_df['Adjusted_date']))
dorm_A_keys = set(zip(dorm_A_q['Subject'], dorm_A_q['Adjusted_date']))
dorm_B_keys = set(zip(dorm_B_q['Subject'], dorm_B_q['Adjusted_date']))

print("Sleep ∩ Dorm A matches:", len(sleep_keys & dorm_A_keys))
print("Sleep ∩ Dorm B matches:", len(sleep_keys & dorm_B_keys))

# --------------------
# Qualtrics columns E–AV
# --------------------
qualtrics_cols = list(dorm_A_q.columns[4:48])

dorm_A_q_subset = dorm_A_q[['Subject', 'Adjusted_date'] + qualtrics_cols]
dorm_B_q_subset = dorm_B_q[['Subject', 'Adjusted_date'] + qualtrics_cols]

# --------------------
# Merge
# --------------------
sleep_dorm_A = sleep_df.merge(
    dorm_A_q_subset,
    on=['Subject', 'Adjusted_date'],
    how='left'
)

sleep_dorm_B = sleep_df.merge(
    dorm_B_q_subset,
    on=['Subject', 'Adjusted_date'],
    how='left'
)

print("\n--- POST-MERGE CHECKS ---")
print("Dorm A merged rows with data:",
      sleep_dorm_A[qualtrics_cols].notna().any(axis=1).sum())
print("Dorm B merged rows with data:",
      sleep_dorm_B[qualtrics_cols].notna().any(axis=1).sum())

# --------------------
# Prepare Qualtrics columns
# --------------------
for col in qualtrics_cols:
    sleep_df[col] = pd.NA

# --------------------
# ROBUST Site matching (THIS IS THE FIX)
# --------------------
sleep_df['Site'] = sleep_df['Site'].astype(str).str.upper()

dorm_A_mask = sleep_df['Site'].str.contains('SITE_A', na=False)
dorm_B_mask = sleep_df['Site'].str.contains('SITE_B', na=False)

print("\n--- SITE CHECK ---")
print("Dorm A rows in sleep_df:", dorm_A_mask.sum())
print("Dorm B rows in sleep_df:", dorm_B_mask.sum())

# --------------------
# Assign Qualtrics data
# --------------------
sleep_df.loc[dorm_A_mask, qualtrics_cols] = (
    sleep_dorm_A.loc[dorm_A_mask, qualtrics_cols].values
)

sleep_df.loc[dorm_B_mask, qualtrics_cols] = (
    sleep_dorm_B.loc[dorm_B_mask, qualtrics_cols].values
)

# --------------------
# QualtricsLogged
# --------------------
sleep_df['QualtricsLogged'] = (
    sleep_df[qualtrics_cols].notna().any(axis=1)
)

print("\n--- FINAL CHECK ---")
print("QualtricsLogged TRUE count:",
      sleep_df['QualtricsLogged'].sum())

# Move 'QualtricsLogged' to column Q (index 16)
cols = [c for c in sleep_df.columns if c != 'QualtricsLogged']
cols.insert(21, 'QualtricsLogged')  # index 16 = column Q
sleep_df = sleep_df[cols]

# --------------------
# Save
# --------------------
# sleep_df.drop(columns=['Site'], inplace=True)
sleep_df.to_csv(output_w_qualtrics_path, index=False)

print("\n✅ Qualtrics data successfully merged and saved.")
print(f"✅ Code has run as of {today_date}, {current_time}")


### 7.3 Attach environmental exposure during each sleep period (file #3)

For each sleep period, indoor (and, for temperature, outdoor) environmental statistics are
computed over the window `InBedTime − 1 hour` through `OutBedTime`, for temperature,
relative humidity, absolute humidity, wet bulb temperature, CO₂,
PM₁, PM₂.₅ and PM₁₀.

In [ ]:
import pandas as pd
import numpy as np
from pandas.api.types import is_datetime64_any_dtype
from datetime import datetime

# -------------------------------------
# Step 1: File paths
# -------------------------------------
sleep_file_path = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics.csv")
output_path = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics-env.csv")

# -------------------------------------
# Step 2: Load sleep data
# -------------------------------------
sleep_df = pd.read_csv(sleep_file_path)

# -------------------------------------
# Step 3: Ensure proper datetime formats (sleep)
# -------------------------------------
# Convert to datetime in default yyyy-mm-dd HH:MM format
sleep_df['InBedTime_1hrbefore'] = pd.to_datetime(
    sleep_df['InBedTime_1hrbefore'], errors='coerce'
)
sleep_df['OutBedTime'] = pd.to_datetime(
    sleep_df['OutBedTime'], errors='coerce'
)

# -------------------------------------
# Step 4: Prepare environmental dataframes
# -------------------------------------
def prepare_env_df(df, site):
    df = df.copy()
    
    # Ensure index is datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors='coerce')
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    
    # Ensure id_room exists
    if 'id_room' not in df.columns:
        raise ValueError(f"❌ 'id_room' missing in {site} dataframe")
    df['id_room'] = df['id_room'].astype(str)
    
    # Sort by datetime index
    df = df.sort_index()
    
    return df

# Example: assume dorm_A_condata_df and dorm_B_condata_df already loaded
dorm_A_env_df = prepare_env_df(dorm_A_condata_df0, site="SITE A")
dorm_B_env_df = prepare_env_df(dorm_B_condata_df0, site="SITE B")

print(dorm_A_condata_df0.columns)
print(dorm_B_condata_df0.columns)

# # Ensure dorm A has co2_corrected even if it wasn't produced by prepare_env_df
# if "co2_corrected" not in dorm_A_env_df.columns and "co2" in dorm_A_env_df.columns:
#     dorm_A_env_df["co2_corrected"] = dorm_A_env_df["co2"]

# -------------------------------------
# Step 5: Environmental variables
# -------------------------------------
env_vars = ['temperature_calibrated', 'humidity_calibrated', 'abs_humidity_c', 'wet_bulb_temp', 'co2_used', 'pm1', 'pm25', 'pm10']

# Keep only variables present in BOTH datasets
env_vars = [v for v in env_vars if v in dorm_A_env_df.columns and v in dorm_B_env_df.columns]
if not env_vars:
    raise ValueError("❌ No common environmental variables found")

# Temperature is the only variable that also gets Outdoor and Indoor-Outdoor stats.
# Site A outdoor readings live under a fixed id_room ("Outside 1"); Site B outdoor
# readings live under "{Room} Outdoor" (mirroring the "{Room} Indoor" id used below).
TEMP_VAR = 'temperature_calibrated'
DORM_A_OUTDOOR_ROOM = "Outside 1"
STAT_SUFFIXES = ['mean', 'std', 'median', 'min', 'max', 'IQR_0.75', 'IQR_0.25']

# -------------------------------------
# Step 6: Helper to compute stats
# -------------------------------------
def compute_env_stats(subset, variables):
    stats = {}
    for var in variables:
        series = subset[var].dropna()
        if series.empty:
            continue
        stats[f'{var}_mean'] = series.mean()
        stats[f'{var}_std'] = series.std()
        stats[f'{var}_median'] = series.median()
        stats[f'{var}_min'] = series.min()
        stats[f'{var}_max'] = series.max()
        stats[f'{var}_IQR_0.75'] = series.quantile(0.75)
        stats[f'{var}_IQR_0.25'] = series.quantile(0.25)
    return pd.Series(stats)

# -------------------------------------
# Step 7: Iterate per sleep period (SITE-AWARE)
# -------------------------------------
results = []

for _, row in sleep_df.iterrows():

    site = row.get('Site')
    room_raw = row.get('Room')

    start = row.get('InBedTime_1hrbefore')
    end = row.get('OutBedTime')

    if pd.isna(site) or pd.isna(room_raw) or pd.isna(start) or pd.isna(end):
        results.append(pd.Series({}))
        continue

    # Select correct environmental dataframe and indoor/outdoor room ids
    if site == "SITE_A":
        env_df = dorm_A_env_df
        indoor_room = room_raw
        outdoor_room = DORM_A_OUTDOOR_ROOM
    elif site == "SITE_B":
        env_df = dorm_B_env_df
        indoor_room = f"{room_raw} Indoor"
        outdoor_room = f"{room_raw} Outdoor"
    else:
        results.append(pd.Series({}))
        continue

    # Subset indoor environment data for that room and period
    indoor_subset = env_df.loc[
        (env_df['id_room'] == indoor_room) &
        (env_df.index >= start) &
        (env_df.index <= end)
    ]

    if indoor_subset.empty:
        results.append(pd.Series({}))
        continue

    stats = compute_env_stats(indoor_subset, env_vars)

    # Outdoor temperature + Indoor-Outdoor difference (temperature only)
    outdoor_subset = env_df.loc[
        (env_df['id_room'] == outdoor_room) &
        (env_df.index >= start) &
        (env_df.index <= end)
    ]

    if not outdoor_subset.empty and TEMP_VAR in outdoor_subset.columns:
        outdoor_stats = compute_env_stats(outdoor_subset, [TEMP_VAR])
        outdoor_stats = outdoor_stats.rename(lambda k: f"{k}_Outdoor")
        stats = pd.concat([stats, outdoor_stats])

        for suffix in STAT_SUFFIXES:
            indoor_key = f"{TEMP_VAR}_{suffix}"
            outdoor_key = f"{TEMP_VAR}_{suffix}_Outdoor"
            if indoor_key in stats and outdoor_key in stats:
                stats[f"{TEMP_VAR}_{suffix}_IndoorOutdoorDiff"] = stats[indoor_key] - stats[outdoor_key]

    results.append(stats)

# -------------------------------------
# Step 8: Merge results back
# -------------------------------------
stats_df = pd.DataFrame(results)
merged_df = pd.concat(
    [sleep_df.reset_index(drop=True), stats_df.reset_index(drop=True)],
    axis=1
)

# -------------------------------------
# Step 9: Ensure all datetime columns in yyyy-mm-dd format
# -------------------------------------
datetime_cols = merged_df.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns
for col in datetime_cols:
    merged_df[col] = merged_df[col].dt.strftime('%Y-%m-%d %H:%M')

# -------------------------------------
# Step 10: Save output
# -------------------------------------
merged_df.to_csv(output_path, index=False)

# -------------------------------------
# Step 11: Summary
# -------------------------------------
today_date = datetime.today().strftime('%Y-%m-%d')
current_time = datetime.now().strftime('%H:%M:%S')

print(f"✅ Code has run as of {today_date}, {current_time}")
print(f"✅ Environmental stats added for variables: {env_vars}")
print(f"✅ Outdoor + Indoor-Outdoor diff stats added for: {TEMP_VAR}")
print(f"✅ Rows processed: {len(merged_df)}")
print(f"✅ Output saved to:\n{output_path}")

### 7.4 Data cleaning & exclusion cascade → final analytic sample

In [ ]:
#updated code
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from pandas.api.types import is_numeric_dtype

# ---------------------------
# File paths
# ---------------------------
previous_file = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics-env.csv")
manuallyremove_file = os.path.join(DATA_DIR, "heats-dorms-manualreviewnights.xlsx")

output_file = os.path.join(DATA_DIR, "heats-dorms-sleepperiodmetrics-1h-qualtrics-env-clean.csv")
removed_rows_output = os.path.join(DATA_DIR, "heats-dorms-condata-rows_removed.csv")

# ---------------------------
# Helper function to log removed rows
# ---------------------------
removed_rows_log = []

def log_removed_rows(df_removed_subset, reason):
    """
    Append removed rows to the global removal log.
    Keeps only Subject, Adjusted_date, Removed_by.
    """
    global removed_rows_log

    if df_removed_subset is None or df_removed_subset.empty:
        return

    cols_needed = ["Subject", "Adjusted_date"]
    missing_cols = [c for c in cols_needed if c not in df_removed_subset.columns]
    if missing_cols:
        raise KeyError(f"Missing required columns for removal log: {missing_cols}")

    temp = df_removed_subset[["Subject", "Adjusted_date"]].copy()
    temp["Adjusted_date"] = pd.to_datetime(temp["Adjusted_date"], errors="coerce").dt.normalize()
    temp["Removed_by"] = reason

    removed_rows_log.append(temp)

# ---------------------------
# Load data
# ---------------------------
df = pd.read_csv(
    previous_file,
    parse_dates=['InBedTime_1hrbefore', 'OutBedTime'],
    dayfirst=True
)

# Standardize Adjusted_date early
df["Adjusted_date"] = pd.to_datetime(df["Adjusted_date"], errors="coerce").dt.normalize()

# ---------------------------
# Define expected columns
# ---------------------------
env_vars = [
    'temperature_calibrated_mean','temperature_calibrated_std','temperature_calibrated_median','temperature_calibrated_min','temperature_calibrated_max',
    'temperature_calibrated_IQR_0.75','temperature_calibrated_IQR_0.25',
    'humidity_calibrated_mean','humidity_calibrated_std','humidity_calibrated_median','humidity_calibrated_min','humidity_calibrated_max','humidity_calibrated_IQR_0.75','humidity_calibrated_IQR_0.25',
    'abs_humidity_c_mean', 'abs_humidity_c_std','abs_humidity_c_median','abs_humidity_c_min','abs_humidity_c_max','abs_humidity_c_IQR_0.75','abs_humidity_c_IQR_0.25',
    'wet_bulb_temp_mean','wet_bulb_temp_std','wet_bulb_temp_median','wet_bulb_temp_min','wet_bulb_temp_max','wet_bulb_temp_IQR_0.75','wet_bulb_temp_IQR_0.25',
    'co2_used_mean','co2_used_std','co2_used_median','co2_used_min','co2_used_max','co2_used_IQR_0.75','co2_used_IQR_0.25',
    'pm25_mean','pm25_std','pm25_median','pm25_min','pm25_max','pm25_IQR_0.75','pm25_IQR_0.25',
]
# ---------------------------
# [1] Remove duplicate rows based on Subject + Adjusted_date
# ---------------------------
# Count rows before removal
rows_before = len(df)

dup_mask = df.duplicated(subset=["Subject", "Adjusted_date"], keep="first")

# Log rows that will be removed
df_removed_duplicates = df.loc[dup_mask, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_duplicates, "duplicated Subject–Adjusted_date rows")

# Remove duplicates
df = df.loc[~dup_mask].copy()

print(f"✅ TOTAL ROWS BEFORE: {rows_before}")
print(f"🚫 {dup_mask.sum()} duplicated rows removed")

# ---------------------------
# [1] Remove rows with 27 Sept's data
# ---------------------------
before = len(df)

mask_remove_27sept = df['Adjusted_date'] == pd.Timestamp('2025-09-27')
df_removed_27sept = df.loc[mask_remove_27sept, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_27sept, "Adjusted_date = 2025-09-27")

df = df.loc[~mask_remove_27sept].copy()

after = len(df)
print(f"\n🚫 {before-after} rows removed with Adjusted_date = 2025-09-27")
print(f"✅ Remaining valid nights for analysis: {len(df)}")


# ---------------------------
# [2] Filter out rows missing environmental data
# ---------------------------
sentinel = 'temperature_calibrated_mean'
excluded_mask = df[sentinel].isna()
excluded_rows = df.loc[excluded_mask, ['Subject', 'Adjusted_date']].copy()

# print("\nRows with missing environmental data:")
# if len(excluded_rows) > 0:
#     display_cols = ['Subject', 'Adjusted_date']
#     extra_display_col = 'Room' if 'Room' in df.columns else None
#     if extra_display_col:
#         print(df.loc[excluded_mask, ['Subject', 'Room', 'Adjusted_date']].to_string(index=False))
#     else:
#         print(excluded_rows.to_string(index=False))
# else:
#     print("None found — all rows contain environmental data.")

log_removed_rows(excluded_rows, "missing environmental data")

df_valid = df.loc[~excluded_mask].copy()
print(f"\n🚫 {len(excluded_rows)} rows removed due to missing environmental data")
print(f"✅ Remaining valid nights for analysis: {len(df_valid)}")

# ---------------------------
# Row count + QualtricsLogged summary
# ---------------------------
total_rows = len(df_valid)

qualtrics_true_mask = df_valid['QualtricsLogged'].astype(str).str.upper() == 'TRUE'
true_rows = qualtrics_true_mask.sum()
false_rows = (~qualtrics_true_mask).sum()

df_removed_qualtrics = df_valid.loc[~qualtrics_true_mask, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_qualtrics, "QualtricsLogged != TRUE")

print(f"\n🚫 {total_rows - true_rows} rows removed due to QualtricsLogged != TRUE")

# ---------------------------
# [3] Remove rows if there are no Qualtrics data
# ---------------------------
df_valid = df_valid.loc[qualtrics_true_mask].copy()
print(f"✅ Remaining valid nights for analysis: {len(df_valid)}")

# -----------------------------
# Columns to check
# -----------------------------
cols_to_check = [
    "InBedTime", "OutBedTime", "Onset",
    "LatencyInMinutes", "AvgAwakeningInMinutes", "AwakeningCount",
    "Efficiency", "SE_recalculated", "TimeAsleepInMinutes",
    "TimeAwakeInMinutes", "WakeAfterOnsetInMinutes",
    "TotalInBedTime_minutes", "TIB_recalculated",
    "InBedTime_1hrbefore", "QualtricsLogged",
    "night_survey_timestamp", "morning_survey_timestamp",
    "Q0.2 BEF / AFT",
    "Q1.11 Workday TODAY", "Q1.12 Workday TMRW",
    "Q1.2", "Q1.3 PRE-RTS", "Q1.4 RTP", "Q1.5 Wind",
    "J_row2", "Q2.1 Wake time", "Q2.2 POST-RTS",
    "Q2.3 POST-RTP", "Q2.4 POST-Wind",
    "Q2.5 Sleep Quality", "Q2.6 Refreshed",
    "INT", "Q2.7 Sleep disrup",
]

# Keep only columns that actually exist
cols_to_check_existing = [col for col in cols_to_check if col in df_valid.columns]

# -----------------------------
# Normalize blanks (NaN, empty, whitespace)
# -----------------------------
df_check = df_valid.copy()

for col in cols_to_check_existing:
    df_check[col] = df_check[col].replace(r"^\s*$", np.nan, regex=True)

# -----------------------------
# Summary counts
# -----------------------------
total_rows = len(df_check)
rows_with_any_blank = df_check[cols_to_check_existing].isna().any(axis=1)
n_rows_with_blanks = rows_with_any_blank.sum()

# Keep only complete rows
df_removed_incomplete = df_valid.loc[rows_with_any_blank, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_incomplete, "incomplete Qualtrics data")

df_valid = df_valid.loc[~rows_with_any_blank].copy()

print(f"\n🚫 {rows_with_any_blank.sum()} rows removed that have incomplete Qualtrics data")
print(f"✅ Remaining rows: {len(df_valid)}")

# ---------------------------
# [4] Remove nights which should be manually removed
# ---------------------------
df_remove = pd.read_excel(manuallyremove_file, usecols=["Subject", "Adjusted_date"])

df_valid["Adjusted_date"] = pd.to_datetime(
    df_valid["Adjusted_date"],
    errors="coerce",
    infer_datetime_format=True
).dt.normalize()

df_remove["Adjusted_date"] = pd.to_datetime(
    df_remove["Adjusted_date"],
    errors="coerce",
    infer_datetime_format=True
).dt.normalize()

assert pd.api.types.is_datetime64_any_dtype(df_valid["Adjusted_date"]), \
    "❌ df_valid Adjusted_date is NOT datetime"

assert pd.api.types.is_datetime64_any_dtype(df_remove["Adjusted_date"]), \
    "❌ df_remove Adjusted_date is NOT datetime"

remove_keys = set(zip(df_remove["Subject"], df_remove["Adjusted_date"]))

df_valid["_to_remove"] = list(zip(df_valid["Subject"], df_valid["Adjusted_date"]))
df_valid["_to_remove"] = df_valid["_to_remove"].isin(remove_keys)

df_removed_manual = df_valid[df_valid["_to_remove"]].drop(columns="_to_remove")
df_ready = df_valid[~df_valid["_to_remove"]].drop(columns="_to_remove")

# Log rows removed from manuallyremove_file
log_removed_rows(df_removed_manual, "manually removed after checking Actograms")

check = df_ready.merge(
    df_remove,
    on=["Subject", "Adjusted_date"],
    how="inner"
)

removed_counts_manual = (
    df_removed_manual
    .groupby("Subject")
    .size()
    .sort_values(ascending=False)
)

total_nights_manual = removed_counts_manual.sum()
total_subjects_manual = removed_counts_manual.shape[0]

print(f"\n🚫 {total_nights_manual} rows removed across {total_subjects_manual} subjects after manually checking Actograms")
print(f"✅ Remaining valid nights for analysis after manual removal: {len(df_ready)}")

# ============================================================
# [5] Sequential ±3SD filtering for extreme sleep values
# Order: TST -> TIB -> SE -> SOL -> WASO
# ============================================================

print("\n" + "=" * 70)
print("EXTREME VALUE CLEANING (±3 SD)")
print("=" * 70)

summary_params = [
    "LatencyInMinutes",            # SOL
    "AwakeningCount",
    "SE_recalculated",             # SE
    "TimeAsleepInMinutes",         # TST
    "WakeAfterOnsetInMinutes",     # WASO
    "TIB_recalculated"             # TIB
]

stats_summary = {}

for col in summary_params:
    if col in df_ready.columns:
        mean_val = df_ready[col].mean()
        sd_val = df_ready[col].std()
        stats_summary[col] = (mean_val, sd_val)
        # print(f"{col:30s}  Mean = {mean_val:.2f}   SD = {sd_val:.2f}")
    else:
        print(f"{col:30s}  ⚠ Column not found")

filter_sequence = [
    ("TimeAsleepInMinutes", "TST"),
    ("TIB_recalculated", "TIB"),
    ("SE_recalculated", "SE"),
    ("LatencyInMinutes", "SOL"),
    ("WakeAfterOnsetInMinutes", "WASO")
]

df_clean = df_ready.copy()
df_clean["Adjusted_date"] = pd.to_datetime(df_clean["Adjusted_date"], errors="coerce").dt.normalize()

initial_nights = len(df_clean)
removed_counts = {}

required_cols = ["Subject", "Adjusted_date"]
missing_required = [c for c in required_cols if c not in df_clean.columns]
if missing_required:
    raise KeyError(f"Missing required columns for removed-rows output: {missing_required}")

for col, label in filter_sequence:
    if col not in df_clean.columns:
        print(f"\n⚠ {label} column ({col}) not found — skipping")
        continue

    mean_val = df_clean[col].mean()
    sd_val = df_clean[col].std()

    lower = mean_val - 3 * sd_val
    upper = mean_val + 3 * sd_val

    before_n = len(df_clean)

    keep_mask = (
        df_clean[col].notna() &
        (df_clean[col] >= lower) &
        (df_clean[col] <= upper)
    )

    removed_this_step = df_clean.loc[~keep_mask, ["Subject", "Adjusted_date"]].copy()
    log_removed_rows(removed_this_step, f"beyond +- 3 SD for {label}")

    df_clean = df_clean.loc[keep_mask].copy()

    after_n = len(df_clean)
    removed_n = before_n - after_n
    removed_counts[label] = removed_n

    print(f"🚫 {removed_n} rows removed beyond +- 3 SD for {label}")
    print(f"✅ Remaining nights {after_n}")

# ------------------------------------------------------------
# [6] Remove 2 subjects' data
# ------------------------------------------------------------
subjects_to_remove = ['CDS025', 'CDS038']

rows_before = len(df_clean)

mask_remove_subjects = df_clean['Subject'].isin(subjects_to_remove)
df_removed_subjects = df_clean.loc[mask_remove_subjects, ['Subject', 'Adjusted_date']].copy()
log_removed_rows(df_removed_subjects, "eliminating CDS025 and CDS038")

df_clean = df_clean.loc[~mask_remove_subjects].copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"\n🚫 {rows_removed} rows removed after eliminating CDS025 and CDS038")
print(f"✅ FINAL rows remaining: {rows_after}")

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------
final_nights = len(df_clean)
total_removed = initial_nights - final_nights

# ------------------------------------------------------------
# Save all removed rows to CSV
# ------------------------------------------------------------
if removed_rows_log:
    df_removed_all = pd.concat(removed_rows_log, ignore_index=True)

    df_removed_all["Adjusted_date"] = pd.to_datetime(
        df_removed_all["Adjusted_date"], errors="coerce"
    ).dt.normalize()

    df_removed_all = df_removed_all.sort_values(
        by=["Subject", "Adjusted_date", "Removed_by"],
        na_position="last"
    ).reset_index(drop=True)

    df_removed_all.to_csv(removed_rows_output, index=False)

    print("\n" + "=" * 70)
    print("ALL REMOVED ROWS SAVED")
    print("=" * 70)
    print(f"Saved to: {removed_rows_output}")
    # print("\nPreview:")
    # print(df_removed_all.head(20))
else:
    df_removed_all = pd.DataFrame(columns=["Subject", "Adjusted_date", "Removed_by"])
    df_removed_all.to_csv(removed_rows_output, index=False)

    print("\nNo rows were removed. Empty file saved:")
    print(removed_rows_output)

# ------------------------------------------------------------
# Save final cleaned dataset
# ------------------------------------------------------------
df_clean.to_csv(output_file, index=False)
print(f"\ndf_clean saved to:\n{output_file}")


## 8. Assumption Checks

### 8.1 Normality Checks (Shapiro–Wilk)

Run on the final analytic sample (`df_clean`, 686 nights / 38 participants).

In [ ]:
from scipy.stats import shapiro

REQ = [
    "Subject", "Room", "TimeAsleepInMinutes",
    "temperature_calibrated_mean", "humidity_calibrated_mean", "abs_humidity_c_mean",
    "wet_bulb_temp_mean", "co2_used_mean", "pm25_mean", "pm1_mean", "pm10_mean",
    "temperature_calibrated_median", "humidity_calibrated_median", "abs_humidity_c_median",
    "wet_bulb_temp_median", "co2_used_median", "pm25_median", "pm1_median", "pm10_median",
]
missing = [c for c in REQ if c not in df_clean.columns]
assert not missing, f"Missing columns: {missing}"

dc = df_clean

numeric_cols = [
    "TimeAsleepInMinutes", "LatencyInMinutes", "Efficiency", "SE_recalculated", "WakeAfterOnsetInMinutes",
    "TotalInBedTime_minutes", "TIB_recalculated",
    "temperature_calibrated_mean", "humidity_calibrated_mean", "abs_humidity_c_mean",
    "wet_bulb_temp_mean", "co2_used_mean", "pm25_mean", "pm1_mean", "pm10_mean",
    "temperature_calibrated_median", "humidity_calibrated_median", "abs_humidity_c_median",
    "wet_bulb_temp_median", "co2_used_median", "pm25_median", "pm1_median", "pm10_median",
]

print("\n— NORMALITY CHECK (Shapiro–Wilk) —")
for col in numeric_cols:
    if col in dc.columns:
        data = dc[col].dropna()
        if len(data) < 3:
            print(f"{col}: Not enough data for normality test")
            continue
        stat, p = shapiro(data)
        normality = "≈ normal" if p > 0.05 else "non-normal"
        print(f"{col}: W={stat:.3f}, p={p:.3f} → {normality}")

### 8.2 Collinearity Among Environmental Exposures

Wet bulb temperature is a deterministic function of temperature and relative humidity, and
absolute humidity is closely related to both — so temperature, absolute humidity and WBT are
close to collinear by construction. This cell quantifies that with pairwise Pearson
correlations (pooled and within-person), variance inflation factors, and R² of WBT on
temperature + RH, matching the Discussion's collinearity statement (*"Temperature and
absolute humidity showed modest correlation (|r| = 0.6)... VIF < 5"*).

In [ ]:
# === Pairwise collinearity among thermal-exposure metrics ===
# Variables: temperature_calibrated_median, humidity_calibrated_median,
#            abs_humidity_c_median, wet_bulb_temperature_median
#
# Purpose: WBT is a deterministic function of T and RH, so T, AH and WBT are
# close to collinear by construction. This script produces a supplementary
# table + figure that lets a reader see these four columns as different views
# of ONE underlying exposure, rather than four separable predictors.
#
# It reports FOUR complementary pieces of evidence, because a bare Pearson
# correlation matrix is descriptive but not, on its own, a full collinearity
# diagnostic (see notes at the bottom of this file / chat response). The first
# two go into Panel A of the supplementary table, the third into Panel B —
# both panels ship together in one CSV / printed table:
#   1) Pooled (raw) pairwise Pearson r  -> the standard "correlation table"
#   2) Within-person pairwise Pearson r -> matches the person-demeaned
#      exposure actually used in the OLS models (df0["<var>_within"])
#   3) Variance Inflation Factors (VIF), pooled and within-person -> the
#      formal multicollinearity diagnostic; a large VIF is the quantitative
#      version of "these are not separable effects"
#   4) R^2 from regressing WBT jointly on T + RH -> quantifies "deterministic
#      function of" numerically (should be ~1.0 if WBT was computed from a
#      formula on the same T/RH fields)
#
# Setup: pip install plotly kaleido
# Static PNG export (write_image) needs a Chrome/Chromium binary. If you're on
# kaleido>=1 and get a "Kaleido requires Chrome" error, run `plotly_get_chrome`
# once (downloads a copy), or point it at an existing Chrome/Chromium install
# already on your machine.

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import plotly.graph_objects as go

PNG_SCALE = 3  # ~300 dpi-equivalent on export (passed per-figure; works across kaleido versions)

# ---------- 0) Prep ----------
EXPOSURE_VARS = [
    "temperature_calibrated_median",
    "humidity_calibrated_median",
    "abs_humidity_c_median",
    "wet_bulb_temp_median",
]
LABELS = {
    "temperature_calibrated_median": "Temperature",
    "humidity_calibrated_median": "Relative humidity",
    "abs_humidity_c_median": "Absolute humidity",
    "wet_bulb_temp_median": "Wet bulb temperature",
}

REQ = ["Subject", "Room"] + EXPOSURE_VARS
missing = [c for c in REQ if c not in df_clean.columns]
assert not missing, f"Missing columns: {missing}"

df0 = df_clean[REQ].dropna().copy()
df0["Subject"] = df0["Subject"].astype("category")
df0["Room"] = df0["Room"].astype("category")

print(f"Analysis dataset -> Nights: {len(df0)}, Participants: {df0['Subject'].nunique()}")

# ---------- 1) Build within-person (person-demeaned) versions ----------
# This mirrors the "_within" covariate used in the actual OLS models, so the
# collinearity claim is checked on the SAME quantity the models regress on
# (raw pooled correlation mixes between-person and within-person variation).
g = df0.groupby("Subject", observed=True)
for v in EXPOSURE_VARS:
    df0[f"{v}_s"] = g[v].transform("median")
    df0[f"{v}_within"] = df0[v] - df0[f"{v}_s"]

within_cols = [f"{v}_within" for v in EXPOSURE_VARS]

# ---------- 2) Helper: pairwise Pearson r + 95% CI (Fisher z) + p ----------
def pairwise_corr_table(data, cols, labels):
    rows = []
    n = len(data)
    for i, a in enumerate(cols):
        for b in cols[i + 1:]:
            r, p = stats.pearsonr(data[a], data[b])
            # Fisher z CI
            z = np.arctanh(r)
            se = 1 / np.sqrt(n - 3)
            lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
            rows.append({
                "Variable 1": labels[a.replace("_within", "")] if a.replace("_within", "") in labels else a,
                "Variable 2": labels[b.replace("_within", "")] if b.replace("_within", "") in labels else b,
                "r": r,
                "95% CI": f"{lo:.2f} to {hi:.2f}",
                "p": p,
                "N": n,
            })
    return pd.DataFrame(rows)

pooled_tbl = pairwise_corr_table(df0, EXPOSURE_VARS, LABELS)
within_tbl = pairwise_corr_table(df0, within_cols, LABELS)

print("\n=== Pooled (raw) pairwise Pearson correlations ===")
print(pooled_tbl.round(3).to_string(index=False))

print("\n=== Within-person (person-demeaned) pairwise Pearson correlations ===")
print(within_tbl.round(3).to_string(index=False))

# ---------- 3) Variance Inflation Factors ----------
# VIF_j = 1 / (1 - R^2_j), where R^2_j comes from regressing variable j on the
# other three. VIF > 5 (conservative) or >10 (lenient) is the usual flag for
# "this variable's effect cannot be separated from the others."
def vif_table(data, cols, labels):
    X = sm.add_constant(data[cols])
    out = []
    for i, c in enumerate(cols):
        v = variance_inflation_factor(X.values, i + 1)  # +1 skips the constant column
        base = c.replace("_within", "")
        out.append({"Variable": labels.get(base, c), "VIF": v})
    return pd.DataFrame(out)

vif_pooled = vif_table(df0, EXPOSURE_VARS, LABELS)
vif_within = vif_table(df0, within_cols, LABELS)

print("\n=== VIF, pooled ===")
print(vif_pooled.round(2).to_string(index=False))

print("\n=== VIF, within-person ===")
print(vif_within.round(2).to_string(index=False))

# ---------- 4) Quantify "WBT is a deterministic function of T and RH" ----------
X = sm.add_constant(df0[["temperature_calibrated_median", "humidity_calibrated_median"]])
y = df0["wet_bulb_temp_median"]
r2_wbt = sm.OLS(y, X).fit().rsquared
print(f"\nR^2 of WBT ~ Temperature + Relative humidity (pooled): {r2_wbt:.4f}")

# ---------- 5) Build a single supplementary-ready table ----------
def star(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

supp_tbl = pooled_tbl.copy()
supp_tbl["r (95% CI)"] = supp_tbl.apply(
    lambda row: f"{row['r']:.2f}{star(row['p'])} ({row['95% CI']})", axis=1
)
supp_tbl_within = within_tbl.copy()
supp_tbl_within["r_within (95% CI)"] = supp_tbl_within.apply(
    lambda row: f"{row['r']:.2f}{star(row['p'])} ({row['95% CI']})", axis=1
)

final_supp = supp_tbl[["Variable 1", "Variable 2", "r (95% CI)"]].merge(
    supp_tbl_within[["Variable 1", "Variable 2", "r_within (95% CI)"]],
    on=["Variable 1", "Variable 2"],
)
final_supp = final_supp.rename(columns={
    "r (95% CI)": "Pooled r (95% CI)",
    "r_within (95% CI)": "Within-person r (95% CI)",
})

# VIF panel: one row per variable (not per pair), pooled and within-person
# side by side, so the reader can see both the pairwise correlations AND the
# formal collinearity diagnostic in one supplementary table.
vif_panel = vif_pooled.rename(columns={"VIF": "Pooled VIF"}).merge(
    vif_within.rename(columns={"VIF": "Within-person VIF"}), on="Variable"
)
vif_panel["Pooled VIF"] = vif_panel["Pooled VIF"].round(1)
vif_panel["Within-person VIF"] = vif_panel["Within-person VIF"].round(1)

print("\n=== Supplementary Table: exposure-metric correlations and collinearity (VIF) ===")
print("\nPanel A — Pairwise Pearson correlations")
print(final_supp.to_string(index=False))
print("*p<0.05, **p<0.01, ***p<0.001 (pooled p-values are descriptive only; "
      "they do not account for repeated measures within participant / clustering — "
      "see notes)")
print("\nPanel B — Variance inflation factors (VIF ≈ 1/(1-R²) from regressing each "
      "variable on the other three; VIF > 5-10 flags severe collinearity)")
print(vif_panel.to_string(index=False))

# Single CSV with both panels stacked, so the whole supplementary table is one
# file: a title row for each panel, then that panel's table, blank line between.
with open(os.path.join(TABLES_DIR, "supp_table_exposure_correlations_and_vif.csv"), "w") as f:
    f.write("Panel A: Pairwise Pearson correlations among thermal exposure metrics\n")
    final_supp.to_csv(f, index=False)
    f.write("\nPanel B: Variance inflation factors (VIF)\n")
    vif_panel.to_csv(f, index=False)
_tbl_path = os.path.join(TABLES_DIR, "supp_table_exposure_correlations_and_vif.csv")
print(f"\nSaved: {_tbl_path}")

# ---------- 6) Heatmap figure for the supplement (Plotly) ----------
# Correlation is a POLARITY quantity (signed, centered on 0), not a plain
# magnitude, so it gets a diverging scale (blue <-> red, neutral gray at 0)
# rather than a single-hue sequential ramp.
corr_matrix = df0[EXPOSURE_VARS].corr(method="pearson")
corr_labels = [LABELS[c] for c in EXPOSURE_VARS]
corr_matrix.index = corr_matrix.columns = corr_labels

DIVERGING_SCALE = [
    [0.00, "#0d366b"],   # r = -1  (deep blue)
    [0.25, "#5598e7"],   # r = -0.5
    [0.50, "#f0efec"],   # r =  0  (neutral gray midpoint)
    [0.75, "#e8837a"],   # r = +0.5
    [1.00, "#8a2420"],   # r = +1  (deep red)
]

z = corr_matrix.values
n = len(corr_labels)
text = np.array([[f"{z[i, j]:.2f}" for j in range(n)] for i in range(n)])
# white text on the darker (high |r|) cells, ink text on the light/gray cells
text_color = np.where(np.abs(z) > 0.55, "#ffffff", "#0b0b0b")

fig = go.Figure(
    data=go.Heatmap(
        z=z,
        x=corr_labels,
        y=corr_labels,
        zmin=-1,
        zmax=1,
        colorscale=DIVERGING_SCALE,
        colorbar=dict(title="Pearson r", tickvals=[-1, -0.5, 0, 0.5, 1]),
        xgap=2,  # surface-gap between cells instead of a border stroke
        ygap=2,
    )
)
# Direct-labeled values on every cell (a small, fully-enumerable grid — this
# is the one case where labeling every mark is the right call, not the
# "never a number on every point" default for larger charts).
annotations = [
    dict(x=corr_labels[j], y=corr_labels[i], text=text[i, j], showarrow=False,
         font=dict(color=text_color[i, j], size=13))
    for i in range(n) for j in range(n)
]
fig.update_layout(
    title="Pairwise correlations among thermal exposure metrics",
    annotations=annotations,
    xaxis=dict(side="bottom", tickangle=-45, showgrid=False),
    yaxis=dict(autorange="reversed", showgrid=False),
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color="#0b0b0b"),
    width=560,
    height=520,
    margin=dict(l=140, r=40, t=60, b=120),
)

fig.show()
# fig.write_image("supp_fig_exposure_correlation_heatmap.png", scale=PNG_SCALE)
# fig.write_html("supp_fig_exposure_correlation_heatmap.html")  # interactive copy, optional
# print("Saved: supp_fig_exposure_correlation_heatmap.png (+ .html)")

# ---------- 7) Scatter matrix (Plotly SPLOM) ----------
# Pearson r only captures linear association, and WBT is a nonlinear function
# of T and RH — a scatter matrix is the visual companion that lets a reader
# see the actual shape of each pairwise relationship, not just its r.
# One relationship per panel (no grouping variable), so this is a single-hue
# mark, not a categorical palette: sequential blue at low opacity (a wash,
# not a saturated block), matching the "1-3 series -> color alone is fine"
# case with n=1 series here.
splom_dims = [dict(label=LABELS[v], values=df0[v]) for v in EXPOSURE_VARS]

fig2 = go.Figure(
    data=go.Splom(
        dimensions=splom_dims,
        showupperhalf=False,   # redundant with the lower triangle; decluttered
        diagonal_visible=False,
        marker=dict(
            color="#256abf",       # sequential blue, mid step
            size=4,                 # r ~ 4 -> >=8px diameter per mark spec
            opacity=0.25,            # wash, not a saturated block
            line=dict(width=0),
        ),
    )
)
fig2.update_layout(
    title="Pairwise relationships among thermal exposure metrics",
    plot_bgcolor="#fcfcfb",
    paper_bgcolor="#fcfcfb",
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color="#0b0b0b"),
    width=750,
    height=750,
)
fig2.update_traces(diagonal_visible=False)
for ax in fig2.layout:
    if ax.startswith("xaxis") or ax.startswith("yaxis"):
        fig2.layout[ax].update(showgrid=True, gridcolor="#e1e0d9", gridwidth=1, zeroline=False)

fig2.show()
# fig2.write_image("supp_fig_exposure_scatter_matrix.png", scale=PNG_SCALE)
# fig2.write_html("supp_fig_exposure_scatter_matrix.html")  # interactive copy, optional
# print("Saved: supp_fig_exposure_scatter_matrix.png (+ .html)")

### 8.3 Linear vs. Non-Linear Exposure–Response Check

Fits a linear and a natural-spline mixed model of TST on temperature (+ absolute humidity,
+ room fixed effects, + subject random intercept) and compares them by AIC/BIC and a
likelihood-ratio test, to check whether a threshold/non-linear relationship would fit
better than the linear specification used in the primary regressions. 

In [ ]:
# ============================================================
# CHECK WHETHER A TEMPERATURE "THRESHOLD" EXISTS
# Approach:
#   1) Fit linear mixed model
#   2) Fit spline mixed model (nonlinear)
#   3) Compare AIC/BIC + Likelihood Ratio Test (LRT)
#   4) Plot spline vs linear + show marginal slope (derivative)
#
# Notes:
# - Uses Subject random intercept + Room fixed effect (C(Room)) for stability.
# - If you insist on Room as random, see the optional block at the bottom.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import chi2

OUTCOME = "TimeAsleepInMinutes"
TEMP = "temperature_calibrated_median"
HUM  = "abs_humidity_c_median"
SUBJ = "Subject"
ROOM = "Room"

# -----------------------------
# 0) Build analysis dataframe
# -----------------------------
dfm = df_clean.copy()
dfm = dfm[[OUTCOME, TEMP, HUM, SUBJ, ROOM]].dropna()
dfm[SUBJ] = dfm[SUBJ].astype("category")
dfm[ROOM] = dfm[ROOM].astype("category")

# Center predictors (helps numerics + interpretability)
dfm["temp_c"] = dfm[TEMP] - dfm[TEMP].mean()
dfm["hum_c"]  = dfm[HUM]  - dfm[HUM].mean()

# Quick counts (useful in Methods)
print("N nights:", len(dfm))
print("N subjects:", dfm[SUBJ].nunique())
print("N rooms:", dfm[ROOM].nunique())

# -----------------------------
# 1) Fit linear mixed model
# -----------------------------
# Subject random intercept; Room controlled as fixed effects
lin_formula = f"{OUTCOME} ~ temp_c + hum_c + C({ROOM})"

m_lin = smf.mixedlm(lin_formula, dfm, groups=dfm[SUBJ])
res_lin = m_lin.fit(reml=False, method="lbfgs", maxiter=500, disp=False)

# -----------------------------
# 2) Fit spline mixed model
# -----------------------------
# df=4 is a common starting choice; try df=3..6 as sensitivity
spl_df = 4
spl_formula = f"{OUTCOME} ~ bs(temp_c, df={spl_df}, degree=3, include_intercept=False) + hum_c + C({ROOM})"

m_spl = smf.mixedlm(spl_formula, dfm, groups=dfm[SUBJ])
res_spl = m_spl.fit(reml=False, method="lbfgs", maxiter=800, disp=False)

print("\n--- Model fit ---")
print("Linear  AIC:", res_lin.aic, " BIC:", getattr(res_lin, "bic", np.nan), " logLik:", res_lin.llf)
print("Spline  AIC:", res_spl.aic, " BIC:", getattr(res_spl, "bic", np.nan), " logLik:", res_spl.llf)
print("ΔAIC (spline - linear):", res_spl.aic - res_lin.aic)

# -----------------------------
# 3) Likelihood Ratio Test (LRT)
# -----------------------------
# Only valid if both models are fitted with ML (reml=False), which we did.
lr_stat = 2 * (res_spl.llf - res_lin.llf)
df_diff = (res_spl.df_modelwc - res_lin.df_modelwc)
p_lrt = chi2.sf(lr_stat, df_diff)

print("\n--- Nonlinearity test (LRT) ---")
print("LR stat:", lr_stat)
print("df diff:", df_diff)
print("p-value:", p_lrt)

# Interpretation guide
print("\n--- Interpretation guide ---")
print("If ΔAIC < 2 and LRT p > 0.05: little evidence of nonlinearity -> threshold unlikely.")
print("If ΔAIC <= -5 (spline much lower AIC) and LRT p < 0.05: evidence of nonlinearity -> threshold/range possible.")

# -----------------------------
# 4) Plot: spline vs linear (publication-style)
# -----------------------------
t_grid = np.linspace(dfm[TEMP].min(), dfm[TEMP].max(), 250)

pred_df = pd.DataFrame({
    TEMP: t_grid,
    "temp_c": t_grid - dfm[TEMP].mean(),
    HUM: dfm[HUM].mean(),
    "hum_c": 0.0,
    # Choose a reference room to condition on; fixed-effects curve depends on room FE.
    # If you want the "average room" curve instead, see note below.
    ROOM: dfm[ROOM].iloc[0],
    SUBJ: dfm[SUBJ].iloc[0],  # dummy
})

yhat_lin = res_lin.predict(pred_df)
yhat_spl = res_spl.predict(pred_df)

# Marginal slope (derivative) from spline curve: minutes per °C
dy_dt = np.gradient(yhat_spl, t_grid)

# Aesthetics similar to your other plot
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 13

plt.figure(figsize=(9, 5), dpi=150)
plt.scatter(dfm[TEMP], dfm[OUTCOME], alpha=0.30, s=14, color="grey", label="Nights")

plt.plot(t_grid, yhat_spl, linewidth=3, color="#FE9B2D", label=f"Spline fit (df={spl_df})")
plt.plot(t_grid, yhat_lin, linewidth=2, color="black", alpha=0.9, label="Linear fit") #linestyle="--",

plt.title("Temperature vs Total Sleep Time (Adjusted for Absolute Humidity)", fontsize=16, fontweight="bold", pad=11)
plt.xlabel("Temperature (°C)")
plt.ylabel("Total Sleep Time (min)")

plt.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))
plt.subplots_adjust(right=0.85)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

# -----------------------------
# 5) Plot marginal slope vs temperature (helps detect a 'threshold')
# -----------------------------
plt.figure(figsize=(9, 4.2), dpi=150)
plt.plot(t_grid, dy_dt, linewidth=3, color="#FE9B2D")
plt.axhline(0, linestyle="--", color="grey", alpha=0.7)

plt.title("Spline marginal effect: minutes per °C across temperature", fontsize=16, fontweight="bold", pad=11)
plt.xlabel("Temperature (°C)")
plt.ylabel("d(Time asleep)/d(Temp) (min per °C)")

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.show()

print("\n--- Summary slope diagnostics ---")
print("Average slope (min/°C):", float(np.mean(dy_dt)))
print("Slope at cool end (first 10%):", float(np.mean(dy_dt[:25])))
print("Slope at warm end (last 10%):", float(np.mean(dy_dt[-25:])))

# ============================================================
# OPTIONAL NOTE:
# The predicted curve above is conditional on the first room's fixed effect.
# If you want an "average across rooms" curve, you can:
#   - Predict for each room and average the predictions at each temperature.
# If you'd like, tell me and I'll provide that averaging code.
# ============================================================

# ============================================================
# OPTIONAL: If you INSIST on Room as random (often less stable)
# Replace C(Room) with vc_formula, and wrap fit in try/except or use df=3.
# ============================================================
# m_spl_vc = smf.mixedlm(
#     f"{OUTCOME} ~ bs(temp_c, df=3, degree=3, include_intercept=False) + hum_c",
#     dfm,
#     groups=dfm[SUBJ],
#     vc_formula={ROOM: f"0 + C({ROOM})"}
# )
# res_spl_vc = m_spl_vc.fit(reml=False, method="lbfgs", maxiter=1000, disp=False)
# print(res_spl_vc.summary())



### Marginal effect plot

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from patsy import dmatrix

TITLE_FONT_SIZE = 26
AXIS_TITLE_SIZE = 30
TICK_FONT_SIZE = 26
LEGEND_FONT_SIZE = 20
RED = "rgb(189,0,38)"

# ---------------------------------------------------------
# Population-average predictions from MixedLM fixed effects
# averaged across rooms, with 95% CI ribbon
# ---------------------------------------------------------

# Grid over observed temperature range
t_grid = np.linspace(dfm[TEMP].min(), dfm[TEMP].max(), 250)
# rooms = list(dfm[ROOM].cat.categories)
rooms = sorted(dfm[ROOM].dropna().unique().tolist())

# Fixed-effect names and covariance
fe_names_spl = list(res_spl.fe_params.index)
fe_names_lin = list(res_lin.fe_params.index)

V_spl = res_spl.cov_params().loc[fe_names_spl, fe_names_spl].to_numpy()
V_lin = res_lin.cov_params().loc[fe_names_lin, fe_names_lin].to_numpy()

beta_spl = res_spl.fe_params.to_numpy()
beta_lin = res_lin.fe_params.to_numpy()

# RHS formulas only
rhs_spl = f"bs(temp_c, df={spl_df}, degree=3, include_intercept=False) + hum_c + C({ROOM})"
rhs_lin = f"temp_c + hum_c + C({ROOM})"

# Build design matrices for each room, then average predictions across rooms
Xspl_rooms = []
Xlin_rooms = []

for r in rooms:
    pred_df = pd.DataFrame({
        TEMP: t_grid,
        "temp_c": t_grid - dfm[TEMP].mean(),
        HUM: dfm[HUM].mean(),
        "hum_c": 0.0,
        ROOM: r
    })

    Xspl_r = dmatrix(rhs_spl, pred_df, return_type="dataframe")
    Xlin_r = dmatrix(rhs_lin, pred_df, return_type="dataframe")

    # Reorder columns to exactly match fitted fixed effects
    Xspl_r = Xspl_r.reindex(columns=fe_names_spl, fill_value=0.0)
    Xlin_r = Xlin_r.reindex(columns=fe_names_lin, fill_value=0.0)

    Xspl_rooms.append(Xspl_r.to_numpy())
    Xlin_rooms.append(Xlin_r.to_numpy())

# Average the design matrix across rooms at each temperature
Xspl_avg = np.mean(np.stack(Xspl_rooms, axis=0), axis=0)   # shape: [n_grid, p]
Xlin_avg = np.mean(np.stack(Xlin_rooms, axis=0), axis=0)

# Predicted mean curves
yhat_spl = Xspl_avg @ beta_spl
yhat_lin = Xlin_avg @ beta_lin

# Standard errors for mean prediction
se_spl = np.sqrt(np.einsum("ij,jk,ik->i", Xspl_avg, V_spl, Xspl_avg))
se_lin = np.sqrt(np.einsum("ij,jk,ik->i", Xlin_avg, V_lin, Xlin_avg))

# 95% CI
z = 1.96
spl_low = yhat_spl - z * se_spl
spl_high = yhat_spl + z * se_spl

lin_low = yhat_lin - z * se_lin
lin_high = yhat_lin + z * se_lin

# ---------------------------------------------------------
# Plotly plot
# ---------------------------------------------------------
fig = go.Figure()

# Scatter points
fig.add_trace(
    go.Scatter(
        x=dfm[TEMP],
        y=dfm[OUTCOME],
        mode="markers",
        name="Nights",
        marker=dict(
            color="grey",
            size=9,
            opacity=0.4
        )
    )
)

# Spline CI ribbon
fig.add_trace(
    go.Scatter(
        x=np.concatenate([t_grid, t_grid[::-1]]),
        y=np.concatenate([spl_high, spl_low[::-1]]),
        fill="toself",
        # fillcolor="rgba(254, 155, 45, 0.22)",
        # line=dict(color="rgba(255,255,255,0)"),
        fillcolor="rgba(189,0,38, 0.22)",
        line=dict(color=RED, width=0),
        hoverinfo="skip",
        showlegend=False
    )
)

# Spline curve
fig.add_trace(
    go.Scatter(
        x=t_grid,
        y=yhat_spl,
        mode="lines",
        name=f"Spline fit (df={spl_df})",
        # line=dict(color="#FE9B2D", width=6)
        line=dict(color=RED, width=6)
    )
)

# Linear curve
fig.add_trace(
    go.Scatter(
        x=t_grid,
        y=yhat_lin,
        mode="lines",
        name="Linear fit",
        line=dict(color="black", width=6)
    )
)

fig.update_layout(
    title=dict(
        text="<b>Association between Dormitory Temperature and Total Sleep Time</b>",
        x=0.5,
        xanchor="center",
        font=dict(size=TITLE_FONT_SIZE, family="Arial", color="black")
    ),
    template="simple_white",
    width=1200,
    height=650,
    margin=dict(r=260),
    font=dict(family="Arial", size=20, color="black"),
    legend=dict(
        x=1.02,
        y=1.0,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(0,0,0,0)",
        font=dict(size=LEGEND_FONT_SIZE, family="Arial", color="black")
    ),
    xaxis=dict(
        title="Dormitory temperature (°C)",
        showline=True,
        linecolor="black",
        linewidth=3,
        mirror=False,
        showgrid=False,
        ticks="outside",
        ticklen=6,
        tickwidth=1
    ),
    yaxis=dict(
        title="Total Sleep Time (min)",
        showline=True,
        linecolor="black",
        linewidth=3,
        mirror=False,
        showgrid=False,
        ticks="outside",
        ticklen=6,
        tickwidth=1
    )
)

fig.show()

## 9. Descriptive Statistics for Nighttime Sleep Periods — Table 1 (n=686 nights)

In [ ]:

# ---- Helpers ----
def mean_iqr(series):
    v = series.dropna().to_numpy()
    return np.nanmean(v), np.nanpercentile(v,25), np.nanpercentile(v,75)

def mean_std(series):
    v = series.dropna().to_numpy()
    return np.nanmean(v), np.nanstd(v)

def median_iqr(series):
    v = series.dropna().to_numpy()
    return np.nanmedian(v), np.nanpercentile(v,25), np.nanpercentile(v,75)

def min_max(series):
    v = series.dropna().to_numpy()
    return np.nanmin(v), np.nanmax(v)

def fmt_num(x, k=2):
    return f"{x:.{k}f}"

def fmt_range(lo, hi, k=2):
    return f"{fmt_num(lo,k)}–{fmt_num(hi,k)}"

def median_p5_p95(series):
    v = series.dropna().to_numpy()
    return np.nanmedian(v), np.nanpercentile(v,5), np.nanpercentile(v,95)

# ---- Counts ----
n_rooms        = dc["Room"].nunique() if "Room" in dc.columns else np.nan
n_participants = dc["Subject"].nunique()
n_nights       = len(dc)
nights_per_subj_mean = dc.groupby("Subject", observed=True)["TimeAsleepInMinutes"].size().mean()
# Nights per participant: mean and range
nights_per_subj_counts = dc.groupby("Subject", observed=True)["TimeAsleepInMinutes"].size()
nights_per_subj_min = nights_per_subj_counts.min()
nights_per_subj_max = nights_per_subj_counts.max()


subject_ids = sorted(dc["Subject"].dropna().unique())

# ---- Environment summaries ----
# Mean and SD
temp_mean, temp_std = mean_std(dc["temperature_calibrated_mean"])
temp_outdoor_mean, temp_outdoor_std = mean_std(dc["temperature_calibrated_mean_Outdoor"])
rh_mean,   rh_std   = mean_std(dc["humidity_calibrated_mean"])
ah_c_mean,   ah_c_std   = mean_std(dc["abs_humidity_c_mean"])
wet_mean, wet_std = mean_std(dc["wet_bulb_temp_mean"])
co2_mean,  co2_std = mean_std(dc["co2_used_mean"])
pm25_mean, pm25_std = mean_std(dc["pm25_mean"])
pm1_mean, pm1_std = mean_std(dc["pm1_mean"])
pm10_mean, pm10_std = mean_std(dc["pm10_mean"])

# Median and IQR
temp_med, temp_q1, temp_q3 = median_iqr(dc["temperature_calibrated_median"])
temp_outdoor_med, temp_outdoor_q1, temp_outdoor_q3 = median_iqr(dc["temperature_calibrated_median_Outdoor"])
rh_med,   rh_q1,   rh_q3   = median_iqr(dc["humidity_calibrated_median"])
ah_c_med,   ah_c_q1,   ah_c_q3   = median_iqr(dc["abs_humidity_c_median"])
wet_med, wet_q1, wet_q3 = median_iqr(dc["wet_bulb_temp_median"])
co2_med,  co2_q1,  co2_q3  = median_iqr(dc["co2_used_median"])
pm25_med, pm25_q1, pm25_q3 = median_iqr(dc["pm25_median"])
pm1_med, pm1_q1, pm1_q3 = median_iqr(dc["pm1_median"])
pm10_med, pm10_q1, pm10_q3 = median_iqr(dc["pm10_median"])

# Median of the medians + 5th/95th
temp_med, temp_p5, temp_p95 = median_p5_p95(dc["temperature_calibrated_median"])
temp_outdoor_med, temp_outdoor_p5, temp_outdoor_p95 = median_p5_p95(dc["temperature_calibrated_median_Outdoor"])
rh_med,   rh_p5,   rh_p95   = median_p5_p95(dc["humidity_calibrated_median"])
ah_c_med,   ah_c_p5,   ah_c_p95   = median_p5_p95(dc["abs_humidity_c_median"])
wet_med, wet_p5, wet_p95 = median_p5_p95(dc["wet_bulb_temp_median"])
co2_med,  co2_p5,  co2_p95  = median_p5_p95(dc["co2_used_median"])
pm25_med, pm25_p5, pm25_p95 = median_p5_p95(dc["pm25_median"])
pm1_med, pm1_p5, pm1_p95    = median_p5_p95(dc["pm1_median"])
pm10_med, pm10_p5, pm10_p95 = median_p5_p95(dc["pm10_median"])

# Min / Max
temp_min, temp_max = min_max(dc["temperature_calibrated_median"])
temp_outdoor_min, temp_outdoor_max = min_max(dc["temperature_calibrated_median_Outdoor"])
rh_min,   rh_max   = min_max(dc["humidity_calibrated_median"])
ah_c_min,   ah_c_max   = min_max(dc["abs_humidity_c_median"])
wet_min, wet_max = min_max(dc["wet_bulb_temp_median"])
co2_min,  co2_max  = min_max(dc["co2_used_median"])
pm25_min, pm25_max = min_max(dc["pm25_median"])
pm1_min, pm1_max = min_max(dc["pm1_median"])
pm10_min, pm10_max = min_max(dc["pm10_median"])

# ---- Sleep summaries ----
# Mean and SD
tst_mean, tst_std = mean_std(dc["TimeAsleepInMinutes"])
sol_mean, sol_std = mean_std(dc["LatencyInMinutes"])
se_mean, se_std   = mean_std(dc["Efficiency"])
se_calc_mean, se_calc_std = mean_std(dc["SE_recalculated"])
waso_mean, waso_std = mean_std(dc["WakeAfterOnsetInMinutes"])
tib_mean, tib_std = mean_std(dc["TotalInBedTime_minutes"])
tib_calc_mean, tib_calc_std = mean_std(dc["TIB_recalculated"])
awakening_mean, awakening_std = mean_std(dc["AwakeningCount"])

# Median and IQR
tst_med,  tst_q1,  tst_q3  = median_iqr(dc["TimeAsleepInMinutes"])
sol_med,  sol_q1,  sol_q3  = median_iqr(dc["LatencyInMinutes"]) 
se_med,   se_q1,   se_q3   = median_iqr(dc["Efficiency"]) 
se_calc_med,   se_calc_q1,   se_calc_q3   = median_iqr(dc["SE_recalculated"])
waso_med, waso_q1, waso_q3 = median_iqr(dc["WakeAfterOnsetInMinutes"]) 
tib_med, tib_q1, tib_q3 = median_iqr(dc["TotalInBedTime_minutes"]) 
tib_calc_med, tib_calc_q1, tib_calc_q3 = median_iqr(dc["TIB_recalculated"]) 
awakening_med, awakening_q1, awakening_q3 = median_iqr(dc["AwakeningCount"])

# Median + 5th/95th
tst_med,  tst_p5,  tst_p95  = median_p5_p95(dc["TimeAsleepInMinutes"])
sol_med,  sol_p5,  sol_p95  = median_p5_p95(dc["LatencyInMinutes"])
se_med,   se_p5,   se_p95   = median_p5_p95(dc["Efficiency"]) 
se_calc_med, se_calc_p5, se_calc_p95 = median_p5_p95(dc["SE_recalculated"]) 
waso_med, waso_p5, waso_p95 = median_p5_p95(dc["WakeAfterOnsetInMinutes"])
tib_med, tib_p5, tib_p95 = median_p5_p95(dc["TotalInBedTime_minutes"]) 
tib_calc_med, tib_calc_p5, tib_calc_p95 = median_p5_p95(dc["TIB_recalculated"])
awakening_med, awakening_p5, awakening_p95 = median_p5_p95(dc["AwakeningCount"])

# Min / Max
tst_min,  tst_max  = min_max(dc["TimeAsleepInMinutes"])
sol_min,  sol_max  = min_max(dc["LatencyInMinutes"]) if "LatencyInMinutes" in dc.columns else (np.nan,np.nan)
se_min,   se_max   = min_max(dc["Efficiency"]) if "Efficiency" in dc.columns else (np.nan,np.nan)
se_calc_min,   se_calc_max   = min_max(dc["SE_recalculated"]) if "SE_recalculated" in dc.columns else (np.nan,np.nan)
waso_min, waso_max = min_max(dc["WakeAfterOnsetInMinutes"]) if "WakeAfterOnsetInMinutes" in dc.columns else (np.nan,np.nan)
tib_min, tib_max = min_max(dc["TotalInBedTime_minutes"]) if "TotalInBedTime_minutes" in dc.columns else (np.nan,np.nan)
tib_calc_min, tib_calc_max = min_max(dc["TIB_recalculated"]) if "TIB_recalculated" in dc.columns else (np.nan,np.nan)
awakening_min, awakening_max = min_max(dc["AwakeningCount"]) if "AwakeningCount" in dc.columns else (np.nan,np.nan)

# # If efficiency appears to be 0–1, convert to %
# if np.nanmedian(dc["Efficiency"]) is not np.nan and np.nanmedian(dc["Efficiency"]) <= 2:
#     se_med, se_q1, se_q3 = se_med*100, se_q1*100, se_q3*100
#     se_min, se_max = se_min*100, se_max*100

# ---- Build paste-ready markdown (unchanged unless you want min/max included) ----
print(f"\n✅ Code has run as of {today_date}, {current_time}")
md = f"""
Across {n_nights} nights (≈ {nights_per_subj_mean:.1f} per participant; range **{nights_per_subj_min}–{nights_per_subj_max}) from {n_participants}** participants residing in {n_rooms} rooms, we observed a 
Mean bedroom temperature of {fmt_num(temp_mean,2)} °C (IQR {fmt_range(temp_q1, temp_q3,2)}), and relative humidity of {fmt_num(rh_mean,0)}% (IQR {fmt_num(rh_q1,0)}–{fmt_num(rh_q3,0)}). 
Median sleep metrics were: TST {fmt_num(tst_med,0)} min (IQR {fmt_num(tst_q1,0)}–{fmt_num(tst_q3,0)}), SOL {fmt_num(sol_med,0)} min (IQR {fmt_num(sol_q1,0)}–{fmt_num(sol_q3,0)}), SE {fmt_num(se_calc_med,0)}% (IQR {fmt_num(se_calc_q1,0)}–{fmt_num(se_calc_q3,0)}), and WASO {fmt_num(waso_med,0)} min (IQR {fmt_num(waso_q1,0)}–{fmt_num(waso_q3,0)}). 
Mean sleep metrics were: TST {fmt_num(tst_mean,0)} min (SD \u00B1 {fmt_num(tst_std,0)}), SOL {fmt_num(sol_mean,0)} min (SD \u00B1 {fmt_num(sol_std,0)}), SE {fmt_num(se_calc_mean,0)}% (SD \u00B1 {fmt_num(se_calc_std,0)}–{fmt_num(se_max,0)}), and WASO {fmt_num(waso_mean,0)} min (SD \u00B1 {fmt_num(waso_std,0)}). 
(Statistics computed on the full ±3 SD exclusion-cascade sample described in Methods → Statistical Analysis; see §8.1 above for the separate, non-excluding LOWESS diagnostic.)
""".strip()

print("\n— COUNTS —")
print(f"Participants: {n_participants} | Rooms: {n_rooms} | Nights: {n_nights} | Nights/participant (mean): {nights_per_subj_mean:.1f}")
print(f"Nights/participant range: {nights_per_subj_min}–{nights_per_subj_max}")
print("Subject IDs:", subject_ids)


# print("\n— ENVIRONMENT means—")
# print(f"Temperature mean {temp_mean:.2f} °C (IQR {temp_q1:.2f}–{temp_q3:.2f}) | Min–Max {temp_min:.2f}–{temp_max:.2f}")
# print(f"Relative humidity mean {rh_mean:.0f}% (IQR {rh_q1:.0f}–{rh_q3:.0f}) | Min–Max {rh_min:.0f}–{rh_max:.0f}")
# print(f"CO2 mean {co2_mean:.0f} ppm (IQR {co2_q1:.0f}–{co2_q3:.0f}) | Min–Max {co2_min:.0f}–{co2_max:.0f}")
# print(f"PM2.5 mean {pm25_mean:.0f} ? (IQR {pm25_q1:.0f}–{pm25_q3:.0f}) | Min–Max {pm25_min:.0f}–{pm25_max:.0f}")
# print(f"PM1.0 mean {pm1_mean:.0f} ? (IQR {pm1_q1:.0f}–{pm1_q3:.0f}) | Min–Max {pm1_min:.0f}–{pm1_max:.0f}")
# print(f"PM10 mean {pm10_mean:.0f} ? (IQR {pm10_q1:.0f}–{pm10_q3:.0f}) | Min–Max {pm10_min:.0f}–{pm10_max:.0f}")


# ---- Printout with 5th–95th percentiles ----
print("\n— FOR TABLE (medians, IQR, 5-95th Percentile), Min-Max —")
print("\n— Nighttime Indoor Environmental Conditions —")
print(f"Temperature median {temp_med:.2f} °C | IQR {temp_q1:.2f}–{temp_q3:.2f} | 5–95th: {temp_p5:.2f}–{temp_p95:.2f} |Min–Max {temp_min:.2f}–{temp_max:.2f}")
print(f"Outdoor temperature median {temp_outdoor_med:.2f} °C | IQR {temp_outdoor_q1:.2f}–{temp_outdoor_q3:.2f} | 5–95th: {temp_outdoor_p5:.2f}–{temp_outdoor_p95:.2f} | Min–Max {temp_outdoor_min:.2f}–{temp_outdoor_max:.2f}")
print(f"Absolute humidity median {ah_c_med:.0f} g/m3 | IQR {ah_c_q1:.0f}–{ah_c_q3:.0f} | 5–95th: {ah_c_p5:.0f}–{ah_c_p95:.0f} | Min–Max {ah_c_min:.0f}–{ah_c_max:.0f}")
print(f"Relative humidity median {rh_med:.0f}% | IQR {rh_q1:.0f}–{rh_q3:.0f} | 5–95th: {rh_p5:.0f}–{rh_p95:.0f} | Min–Max {rh_min:.0f}–{rh_max:.0f}")
print(f"Wet Globe Temp median {wet_med:.2f} °C | IQR {wet_q1:.2f}–{wet_q3:.2f} | 5–95th: {wet_p5:.2f}–{wet_p95:.2f} | Min–Max {wet_min:.2f}–{wet_max:.2f}")
print(f"CO2 median {co2_med:.0f} ppm | IQR {co2_q1:.0f}–{co2_q3:.0f} | 5–95th: {co2_p5:.0f}–{co2_p95:.0f} | Min–Max {co2_min:.0f}–{co2_max:.0f}")
print(f"PM2.5 median {pm25_med:.0f} μg/m3 | IQR {pm25_q1:.0f}–{pm25_q3:.0f} | 5–95th: {pm25_p5:.0f}–{pm25_p95:.0f} | Min–Max {pm25_min:.0f}–{pm25_max:.0f}")

print("\n— Sleep Outcomes —")
print(f"TIB_recalc median {tib_calc_med:.2f} min | IQR {tib_calc_q1:.2f}–{tib_calc_q3:.2f} | 5–95th: {tib_calc_p5:.2f}–{tib_calc_p95:.2f} | Min–Max {tib_calc_min:.2f}–{tib_calc_max:.2f}")
print(f"TST median {tst_med:.0f} min | IQR {tst_q1:.2f}–{tst_q3:.2f} | 5–95th: {tst_p5:.0f}–{tst_p95:.0f} | Min–Max {tst_min:.2f}–{tst_max:.2f}")
print(f"SE median {se_calc_med:.0f}% | IQR {se_calc_q1:.0f}–{se_calc_q3:.0f}% | 5–95th: {se_calc_p5:.0f}–{se_calc_p95:.0f} | Min–Max {se_calc_min:.0f}–{se_calc_max:.0f}%")
print(f"SOL median {sol_med:.0f} min | IQR {sol_q1:.0f}–{sol_q3:.0f} | 5–95th: {sol_p5:.0f}–{sol_p95:.0f} | Min–Max {sol_min:.0f}–{sol_max:.0f}")
print(f"WASO median {waso_med:.0f} min | IQR {waso_q1:.0f}–{waso_q3:.0f} | 5–95th: {waso_p5:.0f}–{waso_p95:.0f} | Min–Max {waso_min:.0f}–{waso_max:.0f}")
print(f"Awakenings median {awakening_med:.0f} | IQR {awakening_q1:.0f}–{awakening_q3:.0f} | 5–95th: {awakening_p5:.0f}–{awakening_p95:.0f} | Min–Max {awakening_min:.0f}–{awakening_max:.0f}")

print(md)


## 10. Primary Analysis — Within-Subject OLS Regressions (Table S1 / Figure 2)

*Methods → Statistical Analysis.* For each of six environmental exposures and five sleep
outcomes, fits

```
outcome ~ (exposure − subject's median exposure) / scale + C(Room)
```

with subject-clustered standard errors, where `scale` rescales CO₂ to "per 100 ppm" and
PM₂.₅ to "per 10 µg/m³" increments (all other exposures use their native 1-unit increment).
Room fixed effects control for room-level characteristics; centering each night's exposure
on the participant's own median isolates the within-subject effect. P-values are
Bonferroni-corrected across the five outcomes, separately within each exposure.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

EXPOSURES = {
    "Temperature": dict(col="temperature_calibrated_median", scale=1, unit="1 °C"),
    "Absolute Humidity": dict(col="abs_humidity_c_median", scale=1, unit="1 g/m³"),
    "Relative Humidity": dict(col="humidity_calibrated_median", scale=1, unit="1%"),
    "Wet Bulb Temperature": dict(col="wet_bulb_temp_median", scale=1, unit="1 °C"),
    "CO2": dict(col="co2_used_median", scale=100, unit="100 ppm"),
    "PM2.5": dict(col="pm25_median", scale=10, unit="10 μg/m³"),
}

OUTCOMES = {
    "TIB": "TIB_recalculated",
    "TST": "TimeAsleepInMinutes",
    "SE": "SE_recalculated",
    "SOL": "LatencyInMinutes",
    "WASO": "WakeAfterOnsetInMinutes",
}

NIGHT_COL = "Adjusted_date"  # per-night date, used later for residual autocorrelation checks


def fit_exposure_models(exposure_col, scale):
    """Fit all 5 outcome models for one environmental exposure."""
    req = ["Subject", "Room", NIGHT_COL, exposure_col] + list(OUTCOMES.values())
    df0 = df_clean[req].dropna().copy()
    df0["Subject"] = df0["Subject"].astype("category")
    df0["Room"] = df0["Room"].astype("category")

    within_col = f"{exposure_col}_within"
    subject_median = df0.groupby("Subject", observed=True)[exposure_col].transform("median")
    df0[within_col] = (df0[exposure_col] - subject_median) / scale

    def fit_outcome(outcome_col):
        formula = f"{outcome_col} ~ {within_col} + C(Room)"
        return smf.ols(formula, data=df0).fit(
            cov_type="cluster",
            cov_kwds={"groups": df0["Subject"], "use_correction": True, "df_correction": True},
        )

    models = {outcome_name: fit_outcome(outcome_col) for outcome_name, outcome_col in OUTCOMES.items()}
    return df0, within_col, models


ols_results = {}
for exposure_name, spec in EXPOSURES.items():
    print(f"\n=== {exposure_name} ({spec['unit']} increment) ===")
    df0, within_col, models = fit_exposure_models(spec["col"], spec["scale"])
    print(f"Analysis dataset -> Nights: {len(df0)}, Participants: {df0['Subject'].nunique()}")

    p_raw = [models[outcome].pvalues[within_col] for outcome in OUTCOMES]
    _, p_bonf, _, _ = multipletests(p_raw, alpha=0.05, method="bonferroni")

    rows = []
    for outcome_name, p_adj in zip(OUTCOMES, p_bonf):
        m = models[outcome_name]
        ci = m.conf_int()
        beta = m.params[within_col]
        lo, hi = ci.loc[within_col, 0], ci.loc[within_col, 1]
        p_this = m.pvalues[within_col]
        rows.append(dict(Outcome=outcome_name, beta=beta, ci_lo=lo, ci_hi=hi, p_raw=p_this, p_bonferroni=p_adj))
        print(f"  {outcome_name}: beta={beta:+.2f} (95% CI {lo:+.2f} to {hi:+.2f}; "
              f"raw p={p_this:.3f}; Bonferroni p={p_adj:.3f})")

    ols_results[exposure_name] = dict(
        column=spec["col"], scale=spec["scale"], unit=spec["unit"],
        df=df0, within_col=within_col, models=models,
        summary=pd.DataFrame(rows).set_index("Outcome"),
    )

print("\nFull statsmodels output for any (exposure, outcome) pair is available via, e.g.:")
print('  ols_results["Temperature"]["models"]["TST"].summary()')

### Residual Diagnostics

In [ ]:
# =====================================================================
# Residual diagnostics across all 5 OLS outcome models for one exposure
# (normality + within-participant autocorrelation across consecutive nights).
# =====================================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import shapiro, norm, probplot
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf



COLOR_DATA = "#2a78d6"
COLOR_REF = "#898781"
COLOR_GRID = "#e1e0d9"
COLOR_TEXT = "#52514e"

# Diagnosed here for Temperature, the manuscript's primary reported exposure; swap the key
# below to check any of the other five (Absolute Humidity, Relative Humidity, Wet Bulb
# Temperature, CO2, PM2.5).
df0 = ols_results["Temperature"]["df"]
models = ols_results["Temperature"]["models"]
night_col = NIGHT_COL

# =====================================================================
# 1) Residual normality — per outcome
# =====================================================================
norm_rows = []
resids = {}
for name, m in models.items():
    r = m.resid.dropna()
    resids[name] = r
    W, p_shapiro = shapiro(r.to_numpy())
    norm_rows.append({
        "Outcome": name,
        "n": len(r),
        "Shapiro W": W,
        "Shapiro p": p_shapiro,
        "Skew": r.skew(),
        "Excess kurtosis": r.kurt(),
    })

norm_summary = pd.DataFrame(norm_rows).sort_values("Shapiro W")
print("— Residual normality by outcome (worst first) —")
print(norm_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# -- Grid of histogram + Q–Q per outcome --
outcome_order = norm_summary["Outcome"].tolist()  # worst-normality first
fig1 = make_subplots(
    rows=len(outcome_order), cols=2,
    subplot_titles=sum([[f"{o} — histogram", f"{o} — Q–Q"] for o in outcome_order], []),
    vertical_spacing=0.04,
)

for i, name in enumerate(outcome_order, start=1):
    r = resids[name].to_numpy()

    counts, bin_edges = np.histogram(r, bins=30)
    bin_width = bin_edges[1] - bin_edges[0]
    fig1.add_trace(
        go.Bar(x=(bin_edges[:-1] + bin_edges[1:]) / 2, y=counts, width=bin_width,
               marker_color=COLOR_DATA, marker_line_width=0, showlegend=False,
               hovertemplate="residual ≈ %{x:.2f}<br>count = %{y}<extra></extra>"),
        row=i, col=1,
    )
    x_grid = np.linspace(r.min(), r.max(), 200)
    normal_curve = norm.pdf(x_grid, r.mean(), r.std()) * len(r) * bin_width
    fig1.add_trace(
        go.Scatter(x=x_grid, y=normal_curve, mode="lines",
                   line=dict(color=COLOR_REF, width=2, dash="dash"),
                   showlegend=False, hoverinfo="skip"),
        row=i, col=1,
    )

    (osm, osr), (slope, intercept, r_fit) = probplot(r, dist="norm")
    fig1.add_trace(
        go.Scatter(x=osm, y=osr, mode="markers",
                   marker=dict(color=COLOR_DATA, size=5),
                   showlegend=False,
                   hovertemplate="theoretical = %{x:.2f}<br>sample = %{y:.2f}<extra></extra>"),
        row=i, col=2,
    )
    qq_x = np.array([osm.min(), osm.max()])
    fig1.add_trace(
        go.Scatter(x=qq_x, y=slope * qq_x + intercept, mode="lines",
                   line=dict(color=COLOR_REF, width=2, dash="dash"),
                   showlegend=False, hoverinfo="skip"),
        row=i, col=2,
    )
    fig1.update_xaxes(gridcolor=COLOR_GRID, row=i, col=1)
    fig1.update_yaxes(gridcolor=COLOR_GRID, row=i, col=1)
    fig1.update_xaxes(gridcolor=COLOR_GRID, row=i, col=2)
    fig1.update_yaxes(gridcolor=COLOR_GRID, row=i, col=2)

fig1.update_layout(
    template="plotly_white", height=280 * len(outcome_order), width=920,
    font=dict(color=COLOR_TEXT),
    margin=dict(t=50, l=60, r=20, b=40),
    title="Residual normality — all outcomes (worst Shapiro-Wilk W first)",
)
# fig1.write_html("residual_normality_all_outcomes.html", include_plotlyjs="cdn")
fig1.show()

# =====================================================================
# 2) Autocorrelation across consecutive nights — per outcome
# =====================================================================
missing = [c for c in ("Subject", night_col) if c not in df0.columns]
if missing:
    hint = [c for c in df0.columns if any(k in c.lower() for k in ("night", "date", "day"))]
    print(
        f"\n[Skipping autocorrelation section] {missing} not in df0.columns.\n"
        f"Add a per-subject night/date column to `REQ` in your OLS block, "
        f"then set `night_col` above to its name.\n"
        f"Columns that look like candidates: {hint}\n"
        f"All df0 columns: {list(df0.columns)}"
    )
else:
    def lag1_autocorr(s):
        s = s.dropna()
        return s.autocorr(lag=1) if len(s) > 2 else np.nan

    ac_rows = []
    acf_by_outcome = {}
    for name, m in models.items():
        r = m.resid
        d = df0.loc[r.index, ["Subject", night_col]].copy()
        d["resid"] = r.values
        d = d.sort_values(["Subject", night_col])

        dw = durbin_watson(d["resid"].dropna())

        ac1_by_subj = d.groupby("Subject", observed=True)["resid"].apply(lag1_autocorr)

        lb_pvals = []
        for _, g in d.groupby("Subject", observed=True):
            rr = g["resid"].dropna()
            if len(rr) > 5:
                lb = acorr_ljungbox(rr, lags=[1], return_df=True)
                lb_pvals.append(lb["lb_pvalue"].iloc[0])
        lb_pvals = np.array(lb_pvals)
        pct_sig = 100 * np.mean(lb_pvals < 0.05) if len(lb_pvals) else np.nan

        ac_rows.append({
            "Outcome": name,
            "Durbin-Watson (global)": dw,
            "Mean lag-1 autocorr": ac1_by_subj.mean(),
            "Min lag-1 autocorr": ac1_by_subj.min(),
            "Max lag-1 autocorr": ac1_by_subj.max(),
            "% subjects Ljung-Box sig (p<.05)": pct_sig,
            "n subjects tested": len(lb_pvals),
        })

        resid_series = d["resid"].dropna().to_numpy()
        n_lags = 10
        acf_by_outcome[name] = acf(resid_series, nlags=n_lags, fft=True)

    ac_summary = pd.DataFrame(ac_rows).sort_values("Mean lag-1 autocorr", ascending=False)
    print("\n— Autocorrelation across consecutive nights, by outcome —")
    print(ac_summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

    # -- ACF bars, one panel per outcome --
    fig2 = make_subplots(rows=1, cols=len(models), subplot_titles=list(models.keys()),
                          shared_yaxes=True)
    n_obs_by_outcome = {name: len(resids[name]) for name in models}
    for i, name in enumerate(models.keys(), start=1):
        vals = acf_by_outcome[name]
        lags = np.arange(len(vals))
        ci_bound = 1.96 / np.sqrt(n_obs_by_outcome[name])
        fig2.add_shape(type="rect", x0=-0.5, x1=len(vals) - 0.5, y0=-ci_bound, y1=ci_bound,
                        fillcolor=COLOR_GRID, opacity=0.6, line_width=0, layer="below",
                        row=1, col=i)
        fig2.add_trace(
            go.Bar(x=lags, y=vals, marker_color=COLOR_DATA, width=0.3, showlegend=False,
                   hovertemplate="lag = %{x}<br>ACF = %{y:.3f}<extra></extra>"),
            row=1, col=i,
        )
        fig2.update_xaxes(title_text="Lag", dtick=1, gridcolor=COLOR_GRID, row=1, col=i)
    fig2.update_yaxes(title_text="Autocorrelation", gridcolor=COLOR_GRID, row=1, col=1)
    fig2.update_layout(
        template="plotly_white", height=380, width=1100,
        font=dict(color=COLOR_TEXT),
        margin=dict(t=60, l=60, r=20, b=50),
        title="Residual ACF by outcome (subject-ordered; shaded = 95% band)",
    )
    # fig2.write_html("residual_acf_all_outcomes.html", include_plotlyjs="cdn")
    fig2.show()

## 11. Forest Plot (Figure 2)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FONT_SIZE = 24
SIG_ALPHA = 0.05


MARKER_SIZE = 18

# ---- Data ------------------------------------------------------------------
# raw (beta, ci_lo, ci_hi, adjusted_p) exactly as reported in the source table.
# CO2 is per 100 ppm increment; PM2.5 is per 10 ug/m3 increment.
ENV_ORDER = [
    "Temperature",
    "Absolute Humidity",
    "Relative Humidity",
    "Wet Bulb Temperature",
    "CO2",
    "PM2.5",
]

ENV_LABELS = {
    "Temperature": "Temperature (1 °C ↑)",
    "Absolute Humidity": "Absolute Humidity (1 g/m³ ↑)",
    "Relative Humidity": "Relative Humidity (1% ↑)",
    "Wet Bulb Temperature": "Wet Bulb Temperature (1 °C ↑)",
    "CO2": "CO₂ (100 ppm ↑)",
    "PM2.5": "PM₂.₅ (10 μg/m³ ↑)",
}

ENV_DATA = {
    exposure: {
        outcome: (row["beta"], row["ci_lo"], row["ci_hi"], row["p_bonferroni"])
        for outcome, row in ols_results[exposure]["summary"].iterrows()
    }
    for exposure in ENV_ORDER
}

# panel_type "loss" negates the coefficient (and CI) so that a reduction in the
# sleep metric plots as a positive "loss" on the x-axis; "increase" plots the
# coefficient as reported.
panels = [
    dict(metric="TIB", panel_type="decrease", x_range=[-18, 11], x_title="Change in Time In Bed (min)"),
    dict(metric="TST", panel_type="decrease", x_range=[-18, 11], x_title="Change in Total Sleep Time (min)"),
    dict(metric="SE", panel_type="decrease", x_range=[-3, 1.4], x_title="Change in Sleep Efficiency (%)"),
    dict(metric="SOL", panel_type="increase", x_range=[-3.6, 2.4], x_title="Change in Sleep Onset Latency (min)"),
    dict(metric="WASO", panel_type="increase", x_range=[-7, 11], x_title="Change in Wake After Sleep Onset (min)"),
]


# ---- Grid layout -------------------------------------------------------------
# 2 rows x 6 columns. Row 1 holds 3 plots, each spanning 2 columns (cols 1-2,
# 3-4, 5-6). Row 2 holds 2 plots, each spanning 2 columns, centered by leaving
# a 1-column pad on either side (pad, 2-3, 4-5, pad).
specs = [
    [{"colspan": 2}, None, {"colspan": 2}, None, {"colspan": 2}, None],
    [None, {"colspan": 2}, None, {"colspan": 2}, None, None],
]
GRID_POSITIONS = [(1, 1), (1, 3), (1, 5), (2, 2), (2, 4)]
PANEL_LABELS = ["(a)", "(b)", "(c)", "(d)", "(e)"]

# ---- Figure ------------------------------------------------------------------
fig = make_subplots(
    rows=2,
    cols=6,
    specs=specs,
    horizontal_spacing=0.175,
    vertical_spacing=0.16,
)

for (row, col), panel in zip(GRID_POSITIONS, panels):
    metric = panel["metric"]
    sign = -1 if panel["panel_type"] == "loss" else 1

    for env in ENV_ORDER:
        beta, ci_lo, ci_hi, p = ENV_DATA[env][metric]

        plotted_beta = sign * beta
        plotted_lo = sign * (ci_hi if sign == -1 else ci_lo)
        plotted_hi = sign * (ci_lo if sign == -1 else ci_hi)

        significant = p < SIG_ALPHA
        marker = dict(
            symbol="diamond" if significant else "circle",
            size=MARKER_SIZE + 2 if significant else MARKER_SIZE,
            color="#D62728" if significant else "black",
        )

        fig.add_trace(
            go.Scatter(
                x=[plotted_beta],
                y=[ENV_LABELS[env]],
                mode="markers+text",
                marker=marker,
                cliponaxis=False,
                text=[f"{plotted_beta:+.1f}"],
                textposition="top center",
                textfont=dict(family="Arial", color="black", size=FONT_SIZE - 2),
                error_x=dict(
                    type="data",
                    symmetric=False,
                    array=[plotted_hi - plotted_beta],
                    arrayminus=[plotted_beta - plotted_lo],
                    color="black",
                    thickness=2,
                    width=6,
                ),
                name=env,
                legendgroup=env,
                showlegend=False,
            ),
            row=row, col=col,
        )

    fig.add_vline(x=0, line=dict(color="gray", width=1, dash="dash"), row=row, col=col)

    fig.update_xaxes(range=panel["x_range"], title_text=panel["x_title"], row=row, col=col)

    n = len(ENV_ORDER)
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=[ENV_LABELS[e] for e in ENV_ORDER][::-1],
        range=[-0.4, n - 1 + 0.9],
        row=row, col=col,
    )

# ---- Panel labels (a)-(e): bold, top-left of each subplot, same font --------
# Subplot axes are numbered in creation order (1 = no suffix, then 2, 3, ...),
# which matches GRID_POSITIONS / panels order above.
# for i, label in enumerate(PANEL_LABELS):
#     suffix = "" if i == 0 else str(i + 1)
#     fig.add_annotation(
#         text=f"<b>{label}</b>",
#         xref=f"x{suffix} domain",
#         yref=f"y{suffix} domain",
#         x=-0.06,
#         y=1.12,
#         xanchor="left",
#         yanchor="top",
#         showarrow=False,
#         font=dict(family="Arial", color="black", size=FONT_SIZE),
#     )

# ---- Global styling: Arial, black text --------------------------------------
fig.update_layout(
    font=dict(family="Arial", color="black", size=FONT_SIZE),
    plot_bgcolor="white",
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(t=100, l=200, r=40, b=80),
    height=1300,
    width=2500,
)

fig.update_xaxes(
    showline=True, linecolor="black", linewidth=2,
    ticks="outside", tickcolor="black",
    zeroline=False,
    tickfont=dict(family="Arial", color="black"),
    title_font=dict(family="Arial", color="black", size=FONT_SIZE),
)
fig.update_yaxes(
    showline=True, linecolor="black", linewidth=2,
    ticks="outside", tickcolor="black",
    tickfont=dict(family="Arial", color="black"),
    title_font=dict(family="Arial", color="black", size=FONT_SIZE),
)

fig.show()
fig.write_image(os.path.join(FIGURES_DIR, "forest_plot_all.png"), width=2100, height=1300, scale=1)
print("done")

## 12. Thermal Comfort & Preference (Figure 3)

*Methods → Subjective Surveys.* Ratings of thermal sensation (7-point ASHRAE scale),
thermal preference and air-movement preference, reported before sleep (n=686). The
manuscript's Figure 3 and text report the **before-sleep** ("BEF") ratings; the after-sleep
("AFT") ratings are also computed below as they were in the original notebook, though they
are not part of the published figure.

In [ ]:
import pandas as pd
import plotly.graph_objects as go

TITLE_SIZE = 30
LABEL_SIZE = 28
PCT_SIZE = 28
LEGEND_SIZE = 22
MIN_WIDTH_FOR_LABEL = 0.08  # 8%

BAR_AREA_WIDTH = 900
BAR_AREA_HEIGHT = 160
LEGEND_AREA_HEIGHT = 220
LEFT_MARGIN, RIGHT_MARGIN, TOP_MARGIN, BOTTOM_MARGIN = 20, 40, 80, 0


def report_category_percentages(df_in, col, categories, title):
    counts = df_in[[col]].dropna()[col].value_counts().reindex(categories).fillna(0)
    total = counts.sum()
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    for cat in categories:
        n = int(counts[cat])
        pct = (n / total * 100) if total > 0 else 0
        print(f"{cat:<18}: {n:>4} ({pct:5.1f}%)")
    print(f"Total N = {int(total)}")


def plot_horizontal_stacked_bar(df_in, col, title, categories, colors, bar_thickness=0.40):
    """One horizontal 100%-stacked bar showing the distribution of `col` over `categories`."""
    report_category_percentages(df_in, col, categories, title)

    counts = df_in[[col]].dropna()[col].value_counts().reindex(categories).fillna(0)
    total = counts.sum()

    fig_width = LEFT_MARGIN + BAR_AREA_WIDTH + RIGHT_MARGIN
    fig_height = TOP_MARGIN + max(BAR_AREA_HEIGHT, LEGEND_AREA_HEIGHT) + BOTTOM_MARGIN
    x_domain_end = BAR_AREA_WIDTH / (fig_width - LEFT_MARGIN - RIGHT_MARGIN)

    fig = go.Figure()
    if total > 0:
        props = counts / total
        for cat in categories:
            val = props[cat]
            if val <= 0:
                continue
            text_label = f"{val*100:.0f}%" if val >= MIN_WIDTH_FOR_LABEL else ""
            fig.add_trace(go.Bar(
                y=[""], x=[val], name=cat, orientation="h", width=bar_thickness,
                marker=dict(color=colors[cat]), text=[text_label], textposition="inside",
                textfont=dict(size=PCT_SIZE, color="black"), insidetextanchor="middle",
                hovertemplate=f"{cat}<br>Count: {int(counts[cat])}<br>Percent: {val*100:.1f}%<extra></extra>",
            ))

    fig.update_layout(
        autosize=False, width=fig_width, height=fig_height, barmode="stack",
        plot_bgcolor="white", paper_bgcolor="white",
        font=dict(family="Arial", size=LABEL_SIZE, color="black"),
        title=dict(text=f"<b>{title}</b>", x=0.4, y=0.9, xanchor="center", font=dict(size=TITLE_SIZE, color="black")),
        margin=dict(l=LEFT_MARGIN, r=RIGHT_MARGIN, t=TOP_MARGIN, b=BOTTOM_MARGIN),
        legend=dict(traceorder="normal", orientation="h", x=0.5, xanchor="center", y=0.74, yanchor="bottom",
                    font=dict(size=LEGEND_SIZE, color="black"), bgcolor="rgba(255,255,255,0)"),
        xaxis=dict(range=[0, 1], domain=[0, x_domain_end], showgrid=False, showticklabels=False,
                   zeroline=False, fixedrange=True),
        yaxis=dict(domain=[0, 1], showgrid=False, showticklabels=False, zeroline=False, fixedrange=True),
    )
    fig.show()
    # No return value -- see the note at the end of plot_profile() above.

### 12.1 Thermal Sensation (Figure 3a)

In [ ]:
df = df_clean.copy()
for col in ["Q1.3 PRE-RTS"]:
    df[col] = df[col].astype(str).str.strip()

cats_rts = ["Cold", "Cool", "Slightly cool", "Neutral", "Slightly warm", "Warm", "Hot"]
cols_rts = {
    "Hot": "#a7514e", "Warm": "#db735f", "Slightly warm": "#ecadb7", "Neutral": "#cee0b5",
    "Slightly cool": "#c1e3f5", "Cool": "#91c4e9", "Cold": "#728cd0",
}

plot_horizontal_stacked_bar(df, "Q1.3 PRE-RTS", "BEF-Sleep Ratings of Thermal Sensation", cats_rts, cols_rts)

### 12.2 Thermal Preference (Figure 3b)

In [ ]:
df = df_clean.copy()
for col in ["Q1.4 RTP"]:
    df[col] = df[col].astype(str).str.strip()

cats_rtp = ["Warmer", "Without change", "Cooler"]
cols_rtp = {"Cooler": "#db735f", "Without change": "#cce1b6", "Warmer": "#91c4e9"}

plot_horizontal_stacked_bar(df, "Q1.4 RTP", "BEF-Sleep Ratings of Thermal Preference", cats_rtp, cols_rtp)

### 12.3 Air Movement Preference (Figure 3c)

In [ ]:
df = df_clean.copy()
label_map = {"More wind": "More air movement", "No change": "No change", "Less wind": "Less air movement"}
for col in ["Q1.5 Wind"]:
    df[col] = df[col].astype(str).str.strip().replace(label_map)

cats_amp = ["Less air movement", "No change", "More air movement"]
cols_amp = {"More air movement": "#db735f", "No change": "#cce1b6", "Less air movement": "#91c4e9"}

plot_horizontal_stacked_bar(df, "Q1.5 Wind", "BEF-Sleep Ratings of Air Movement Preference", cats_amp, cols_amp)